# 🔏 Swin Transformer Invisible Watermarking
**Architecture:** Swin Transformer Encoder + Frequency Enhancement + CNN Decoder  
**Dataset:** DIV2K (128×128 patches)  
**Message:** 64-bit payload with optional BCH error correction  
**Checkpoints:** Best-ever model + last checkpoint saved to Google Drive (2 files only)  
**Logging:** Visual collages saved to Drive every N batches  

## 📦 Cell 1 — Install Dependencies

In [ ]:
!pip install timm bchlib -q
print("✅ Dependencies installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 3.9 MB/s eta 0:00:00
✅ Dependencies installed.


## ⚙️ Cell 2 — Imports & Device

In [ ]:
import os, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import timm
from google.colab import drive

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


Using device: cuda
GPU: Tesla T4


## 💾 Cell 3 — Mount Google Drive

In [89]:
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/SwinWatermark/'
os.makedirs(DRIVE_DIR, exist_ok=True)
BEST_CKPT = os.path.join(DRIVE_DIR, 'best_model/best_model.pth')
LAST_CKPT = os.path.join(DRIVE_DIR, 'last_checkpoint.pth')
LOG_DIR   = os.path.join(DRIVE_DIR, 'collages/')
os.makedirs(LOG_DIR, exist_ok=True)
print(f'Drive ready  →  {DRIVE_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive ready  →  /content/drive/MyDrive/SwinWatermark/


## 🔧 Cell 4 — Configuration

In [ ]:
# ─── Toggle BCH Error Correction ─────────────────────────────────
USE_BCH = False  # Set True to enable BCH(127,64) error correction
# ──────────────────────────────────────────────────────────────────

DATA_BITS = 64

if USE_BCH:
    import bchlib
    BCH_POLYNOMIAL = 137
    BCH_BITS       = 10
    _bch_tmp = bchlib.BCH(BCH_BITS, prim_poly=BCH_POLYNOMIAL)
    _ecc_tmp = _bch_tmp.encode(bytes(8))
    MESSAGE_LENGTH = (8 + len(_ecc_tmp)) * 8
    print(f'BCH ON  →  data bits: {DATA_BITS}  |  codeword bits: {MESSAGE_LENGTH}')
else:
    MESSAGE_LENGTH = DATA_BITS
    print(f'BCH OFF →  message bits: {MESSAGE_LENGTH}')

# ── Training hyper-parameters ─────────────────────────────────────
IMG_SIZE   = 128
BATCH_SIZE = 16
EPOCHS     = 100
LR         = 1e-4
LR_MIN     = 1e-6

# ── Loss weights ─────────────────────────────────────────────────
LAMBDA_IMG  = 0.5    # was 1.0 — loosen the image quality leash
LAMBDA_MSG  = 10.0    # was 3.0 — make message recovery the priority
LAMBDA_ADV  = 0.001  # was 0.01 — adversary is a distraction at epoch 1
LAMBDA_FREQ = 0.05    # was 0.5 — same reason

# ── Logging intervals ─────────────────────────────────────────────
LOG_EVERY_N_BATCHES = 200
PRINT_EVERY         = 20

print('Config loaded ✅')


BCH OFF →  message bits: 64
Config loaded ✅


## 🔐 Cell 5 — BCH Encode / Decode Helpers

In [ ]:
if USE_BCH:
    import bchlib
    _bch = bchlib.BCH(BCH_BITS, prim_poly=BCH_POLYNOMIAL)

    def bch_encode_bits(bit_tensor):
        """(B, DATA_BITS) float → (B, MESSAGE_LENGTH) float"""
        B, out = bit_tensor.shape[0], []
        for b in range(B):
            bits = bit_tensor[b].detach().cpu().numpy().astype(np.uint8)
            data_bytes = np.packbits(bits).tobytes()
            ecc = _bch.encode(data_bytes)
            full_bits = np.unpackbits(np.frombuffer(data_bytes + ecc, dtype=np.uint8))
            out.append(full_bits[:MESSAGE_LENGTH])
        return torch.tensor(np.stack(out), dtype=torch.float32).to(bit_tensor.device)

    def bch_decode_bits(codeword_tensor):
        """(B, MESSAGE_LENGTH) float → (B, DATA_BITS) float"""
        B, out = codeword_tensor.shape[0], []
        for b in range(B):
            bits = (codeword_tensor[b] > 0.5).cpu().numpy().astype(np.uint8)
            pad  = (8 - len(bits) % 8) % 8
            bits = np.concatenate([bits, np.zeros(pad, dtype=np.uint8)])
            raw  = np.packbits(bits).tobytes()
            data_b, ecc_b = raw[:8], raw[8:8+len(_bch.encode(raw[:8]))]
            try:
                _, data_corr, _ = _bch.decode(data_b, ecc_b)
            except Exception:
                data_corr = data_b
            dec = np.unpackbits(np.frombuffer(data_corr, dtype=np.uint8))[:DATA_BITS]
            out.append(dec.astype(np.float32))
        return torch.tensor(np.stack(out), dtype=torch.float32).to(codeword_tensor.device)

    print(f'BCH helpers ready. Codeword: {MESSAGE_LENGTH} bits')
else:
    bch_encode_bits = bch_decode_bits = None
    print('BCH disabled — using raw 64-bit messages')


BCH disabled — using raw 64-bit messages


## 🗃️ Cell 6 — Download DIV2K Dataset

In [ ]:
!mkdir -p /content/div2k
!wget -q --show-progress -O /content/div2k/DIV2K_train_HR.zip \
    http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
!cd /content/div2k && unzip -q DIV2K_train_HR.zip
print('DIV2K ready ✅')


/content/div2k/DIV2 100%[===================>]   3.29G  8.25MB/s    in 18m 27s 
DIV2K ready ✅


## 🗃️ Cell 7 — DIV2K Dataset & DataLoader

In [ ]:
class DIV2KDataset(Dataset):
    """Random 128x128 crops from DIV2K HR images."""
    def __init__(self, root, patch_size=128, augment=True):
        self.files = sorted([
            os.path.join(root, f) for f in os.listdir(root)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        self.patch_size = patch_size
        self.augment    = augment
        self.to_tensor  = T.ToTensor()

    def __len__(self):
        return len(self.files) * 20

    def __getitem__(self, idx):
        img = Image.open(self.files[idx % len(self.files)]).convert('RGB')
        w, h = img.size
        p = self.patch_size
        if w < p or h < p:
            img = img.resize((max(w,p), max(h,p)), Image.BICUBIC)
            w, h = img.size
        x0 = random.randint(0, w-p)
        y0 = random.randint(0, h-p)
        patch = img.crop((x0, y0, x0+p, y0+p))
        if self.augment:
            if random.random() > 0.5: patch = TF.hflip(patch)
            if random.random() > 0.5: patch = TF.vflip(patch)
        return self.to_tensor(patch)


DIV2K_ROOT   = '/content/div2k/DIV2K_train_HR'
train_ds     = DIV2KDataset(DIV2K_ROOT, patch_size=IMG_SIZE, augment=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)
print(f'Dataset: {len(train_ds.files)} HR images  →  {len(train_ds)} patches')
print(f'Loader : {len(train_loader)} batches/epoch')


## 🌊 Cell 8 — Frequency Enhancement Module

In [ ]:
class FrequencyEnhancement(nn.Module):
    """
    Decomposes feature maps into low/high frequency bands,
    processes each with a learnable gate, then merges.
    Encourages the watermark to live in perceptually-safe mid-frequencies.
    """
    def __init__(self, channels):
        super().__init__()
        self.low  = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels), nn.GELU()
        )
        self.high = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels), nn.GELU()
        )
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, channels), nn.Sigmoid()
        )
        self.proj = nn.Conv2d(channels*2, channels, 1)

    def forward(self, x):
        lf  = self.low(x)
        hf  = self.high(x - lf)
        g   = self.gate(x).unsqueeze(-1).unsqueeze(-1)
        out = self.proj(torch.cat([lf*g, hf*(1-g)], dim=1))
        return out + x   # residual

print('FrequencyEnhancement module defined ✅')


FrequencyEnhancement module defined ✅


## 🧠 Cell 9 — Swin Transformer Encoder

In [ ]:
class MessageProjection(nn.Module):
    """Projects flat message bits into a spatial feature volume."""
    def __init__(self, msg_len, channels, spatial):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(msg_len, channels * spatial * spatial),
            nn.Unflatten(1, (channels, spatial, spatial))
        )
    def forward(self, msg):
        return self.net(msg)


class SwinEncoder(nn.Module):
    """
    Swin Transformer-based watermark encoder.

    Pipeline:
      1. Swin-T backbone extracts multi-scale features from cover image.
      2. Message bits are projected into spatial volumes at each scale.
      3. Image features + message volumes are fused at 4 scales.
      4. FrequencyEnhancement steers perturbation to mid-frequencies.
      5. Hierarchical upsampling reconstructs full-resolution residual.
      6. Residual is added to cover (constrained by learnable scale).
    """
    FUSION_C = 128

    def __init__(self, msg_len=MESSAGE_LENGTH, img_size=128):
        super().__init__()
        C = self.FUSION_C

        # Swin-Tiny backbone (pretrained, img_size adaptable)
        self.swin = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=True, img_size=img_size,
            features_only=True, out_indices=(0,1,2,3)
        )

        swin_chs   = [96, 192, 384, 768]
        swin_spats = [img_size//4, img_size//8, img_size//16, img_size//32]

        self.reduce  = nn.ModuleList([
            nn.Sequential(nn.Conv2d(c, C, 1), nn.BatchNorm2d(C), nn.GELU())
            for c in swin_chs
        ])
        # self.msg_proj = nn.ModuleList([
        #     MessageProjection(msg_len, C, s) for s in swin_spats
        # ])
        self.fuse = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(C + msg_len, C, 3, padding=1),
                nn.BatchNorm2d(C), nn.GELU(),
                FrequencyEnhancement(C)
            ) for _ in range(4)
        ])

        def up_block(in_c, out_c):
            return nn.Sequential(
                nn.ConvTranspose2d(in_c, out_c, 4, stride=2, padding=1),
                nn.BatchNorm2d(out_c), nn.GELU()
            )

        self.up4 = up_block(C, C)
        self.up3 = up_block(C*2, C)
        self.up2 = up_block(C*2, C)
        self.up1 = up_block(C*2, C)

        self.refine = nn.Sequential(
            FrequencyEnhancement(C),
            nn.Conv2d(C, C, 3, padding=1), nn.BatchNorm2d(C), nn.GELU(),
            nn.Conv2d(C, 3, 1), nn.Tanh()
        )
        self.residual_scale = nn.Parameter(torch.tensor(0.05))

    def forward(self, cover, msg):
        feats = self.swin(cover)
        # Normalise to NCHW (some timm versions return NHWC)
        feats = [
            f.permute(0,3,1,2).contiguous()
            if (f.dim()==4 and f.shape[1]!=f.shape[-1]) else f
            for f in feats
        ]
        red = [self.reduce[i](feats[i]) for i in range(4)]

        fused = []
        for i in range(4):
            h, w = red[i].shape[2:]
            # simple copy: [B, msg_len] → [B, msg_len, 1, 1] → [B, msg_len, H, W]
            mv = msg.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, h, w)
            fused.append(self.fuse[i](torch.cat([red[i], mv], dim=1)))

        x = self.up4(fused[3])
        x = self.up3(torch.cat([x, fused[2]], dim=1))
        x = self.up2(torch.cat([x, fused[1]], dim=1))
        x = self.up1(torch.cat([x, fused[0]], dim=1))

        if x.shape[2:] != cover.shape[2:]:
            x = F.interpolate(x, size=cover.shape[2:],
                              mode='bilinear', align_corners=False)

        residual = self.refine(x)
        scale    = torch.sigmoid(self.residual_scale) * 0.1
        return torch.clamp(cover + residual * scale, 0.0, 1.0)


encoder = SwinEncoder(msg_len=MESSAGE_LENGTH, img_size=IMG_SIZE).to(device)
print(f'Encoder params: {sum(p.numel() for p in encoder.parameters() if p.requires_grad):,}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at /pytorch/aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)
/usr/local/lib/python3.12/dist-packages/timm/layers/interpolate.py:65: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  numerator += self.values[as_s] * \


Encoder params: 30,996,478


## 🔍 Cell 10 — CNN Decoder

In [ ]:
class ConvDecoder(nn.Module):
    """
    Strided CNN decoder with FrequencyEnhancement blocks.
    Predicts soft bit probabilities from the watermarked image.
    """
    def __init__(self, msg_len=MESSAGE_LENGTH):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.GELU(),
            FrequencyEnhancement(64),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.GELU(),
            FrequencyEnhancement(128),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.GELU(),
            FrequencyEnhancement(256),
            nn.Conv2d(256, 512, 3, stride=2, padding=1),
            nn.BatchNorm2d(512), nn.GELU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
        self.head = nn.Sequential(
            nn.Linear(512, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, msg_len), nn.Sigmoid()
        )

    def forward(self, x):
        return self.head(self.body(x))


decoder = ConvDecoder(msg_len=MESSAGE_LENGTH).to(device)
print(f'Decoder params: {sum(p.numel() for p in decoder.parameters() if p.requires_grad):,}')


Decoder params: 2,143,296


## 🕵️ Cell 11 — PatchGAN Discriminator

In [ ]:
class PatchDiscriminator(nn.Module):
    """PatchGAN — classifies real vs watermarked at patch level."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 1, 4, padding=1),
        )
    def forward(self, x):
        return self.net(x)


discriminator = PatchDiscriminator().to(device)
print(f'Discriminator params: {sum(p.numel() for p in discriminator.parameters()):,}')


Discriminator params: 663,745


## 🔊 Cell 12 — Differentiable Noise / Augmentation Layer

In [ ]:
class NoiseLayer(nn.Module):
    """
    Applies random distortions during training to build robustness.
    Randomly picks: Gaussian noise | color jitter | Gaussian blur | identity.
    """
    def forward(self, x):
      return x;
        # if not self.training:
        #     return x
        # r = random.random()
        # if r < 0.25:
        #     return torch.clamp(x + 0.03*torch.randn_like(x), 0, 1)
        # elif r < 0.50:
        #     return T.ColorJitter(brightness=0.1, contrast=0.1)(x)
        # elif r < 0.75:
        #     return T.GaussianBlur(3, sigma=(0.1, 1.0))(x)
        # else:
        #     return x


noise_layer = NoiseLayer().to(device)
print('Noise layer ready ✅')


Noise layer ready ✅


## 📐 Cell 13 — Loss Functions & Metrics

In [ ]:
def ssim_loss(x, y, w=11, C1=1e-4, C2=9e-4):
    """Differentiable 1-SSIM loss."""
    mu_x = F.avg_pool2d(x, w, 1, w//2)
    mu_y = F.avg_pool2d(y, w, 1, w//2)
    sx   = F.avg_pool2d(x*x, w, 1, w//2) - mu_x**2
    sy   = F.avg_pool2d(y*y, w, 1, w//2) - mu_y**2
    sxy  = F.avg_pool2d(x*y, w, 1, w//2) - mu_x*mu_y
    ssim = ((2*mu_x*mu_y+C1)*(2*sxy+C2))/((mu_x**2+mu_y**2+C1)*(sx+sy+C2))
    return 1 - ssim.mean()


def freq_loss(cover, encoded):
    """Penalise high-frequency differences (Laplacian)."""
    k = torch.tensor([[[[0,-1,0],[-1,4,-1],[0,-1,0]]]],
                     dtype=torch.float32, device=cover.device).expand(3,1,3,3)
    return F.mse_loss(F.conv2d(encoded, k, padding=1, groups=3),
                      F.conv2d(cover,   k, padding=1, groups=3))


def bit_accuracy(pred, target):
    return ((pred > 0.5).float() == target).float().mean().item()


def psnr(cover, encoded):
    mse = F.mse_loss(cover, encoded).item()
    return float('inf') if mse == 0 else 10*math.log10(1.0/mse)


bce_loss = nn.BCELoss()
mse_loss = nn.MSELoss()
adv_bce  = nn.BCEWithLogitsLoss()
print('Loss functions ready ✅')


Loss functions ready ✅


## 🚀 Cell 14 — Optimisers & LR Schedulers

In [ ]:
opt_enc_dec = optim.AdamW(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=LR, weight_decay=1e-4
)
opt_disc = optim.AdamW(discriminator.parameters(), lr=LR*0.5, weight_decay=1e-4)

sched_enc_dec = optim.lr_scheduler.CosineAnnealingLR(opt_enc_dec, T_max=EPOCHS, eta_min=LR_MIN)
sched_disc    = optim.lr_scheduler.CosineAnnealingLR(opt_disc,    T_max=EPOCHS, eta_min=LR_MIN)
print('Optimisers ready ✅')


Optimisers ready ✅


## 💾 Cell 15 — Checkpoint Utilities

In [90]:
# Only initialise if not already set (survives cell re-runs)
if 'history' not in dir() or history is None:
    history     = {'epoch':[], 'msg_loss':[], 'img_loss':[], 'acc':[], 'psnr':[]}
if 'best_acc' not in dir():
    best_acc    = 0.0
if 'start_epoch' not in dir():
    start_epoch = 0


def save_checkpoint(path, epoch, is_best=False):
    torch.save({
        'epoch'         : epoch,
        'encoder'       : encoder.state_dict(),
        'decoder'       : decoder.state_dict(),
        'discriminator' : discriminator.state_dict(),
        'opt_enc_dec'   : opt_enc_dec.state_dict(),
        'opt_disc'      : opt_disc.state_dict(),
        'sched_enc_dec' : sched_enc_dec.state_dict(),
        'sched_disc'    : sched_disc.state_dict(),
        'history'       : history,
        'best_acc'      : best_acc,
        'use_bch'       : USE_BCH,
        'msg_len'       : MESSAGE_LENGTH,
    }, path)
    tag = '🏆 BEST' if is_best else '💾 LAST'
    print(f'  {tag} checkpoint → {os.path.basename(path)}')


def load_checkpoint(path):
    global best_acc, start_epoch, history

    if not os.path.exists(path):
        print(f'  No checkpoint found at {path}. Starting fresh.')
        return False

    ckpt = torch.load(path, map_location=device)

    # ── Compatibility check — reject mismatched arch ──────────────────
    if ckpt.get('msg_len') != MESSAGE_LENGTH:
        print(f'  ⚠️  msg_len mismatch ({ckpt.get("msg_len")} vs {MESSAGE_LENGTH}). Deleting stale checkpoint.')
        os.remove(path)
        return False
    if ckpt.get('use_bch') != USE_BCH:
        print(f'  ⚠️  USE_BCH mismatch ({ckpt.get("use_bch")} vs {USE_BCH}). Deleting stale checkpoint.')
        os.remove(path)
        return False

    # ── Load models ───────────────────────────────────────────────────
    try:
        encoder.load_state_dict(ckpt['encoder'])
        decoder.load_state_dict(ckpt['decoder'])
        discriminator.load_state_dict(ckpt['discriminator'])
    except RuntimeError as e:
        print(f'  ⚠️  Model weight mismatch — arch changed. Deleting stale checkpoint.')
        print(f'      {str(e)[:120]}')
        os.remove(path)
        return False

    # ── Load optimisers & schedulers ──────────────────────────────────
    opt_enc_dec.load_state_dict(ckpt['opt_enc_dec'])
    opt_disc.load_state_dict(ckpt['opt_disc'])
    sched_enc_dec.load_state_dict(ckpt['sched_enc_dec'])
    sched_disc.load_state_dict(ckpt['sched_disc'])

    # ── Restore training state ────────────────────────────────────────
    history.clear()
    history.update(ckpt['history'])
    best_acc    = ckpt['best_acc']
    start_epoch = ckpt['epoch'] + 1

    print(f'  ✅ Resumed from epoch {ckpt["epoch"]}  '
          f'(best acc={best_acc*100:.2f}%  next epoch={start_epoch})')
    return True


# ── Auto-resume from last checkpoint ─────────────────────────────────
load_checkpoint(BEST_CKPT)

  ✅ Resumed from epoch 29  (best acc=87.79%  next epoch=30)


True

## 🖼️ Cell 16 — Collage Logger (Drive)

In [ ]:
def bits_to_hex(bits_tensor):
    bits = (bits_tensor > 0.5).int().cpu().numpy().astype(np.uint8)
    pad  = (8 - len(bits)%8) % 8
    bits = np.concatenate([bits, np.zeros(pad, dtype=np.uint8)])
    return ''.join(f'{b:02x}' for b in np.packbits(bits))


def save_collage(cover, encoded, msg, decoded_msg, epoch, batch_idx, mean_acc):
    """
    Saves a 4-sample visual collage to Google Drive.
    Columns: Cover | Watermarked | Difference×10
    Footer : MSG hex | Decoded hex | per-sample bit accuracy
    """
    n = min(4, cover.size(0))
    fig, axes = plt.subplots(n, 3, figsize=(12, n*4))
    if n == 1: axes = axes[np.newaxis, :]

    for i in range(n):
        cov  = np.clip(cover[i].cpu().permute(1,2,0).numpy(), 0, 1)
        enc  = np.clip(encoded[i].detach().cpu().permute(1,2,0).numpy(), 0, 1)
        diff = np.clip((enc - cov)*10 + 0.5, 0, 1)

        m_hex = bits_to_hex(msg[i])
        d_hex = bits_to_hex(decoded_msg[i])
        acc_i = ((decoded_msg[i] > 0.5).float() == msg[i]).float().mean().item()
        p_db  = psnr(cover[i:i+1], encoded[i:i+1])

        for j, (im, title) in enumerate([
            (cov,  f'Cover #{i}'),
            (enc,  f'Watermarked  PSNR={p_db:.1f}dB'),
            (diff, 'Difference x10')
        ]):
            axes[i,j].imshow(im)
            axes[i,j].set_title(title, fontsize=8, pad=3)
            axes[i,j].axis('off')

        color = 'green' if acc_i > 0.95 else 'darkorange' if acc_i > 0.85 else 'red'
        axes[i,0].set_xlabel(f'MSG: 0x{m_hex[:16]}...', fontsize=7, color='navy')
        axes[i,2].set_xlabel(f'DEC: 0x{d_hex[:16]}...  Acc={acc_i:.3f}',
                             fontsize=7, color=color)

    fig.suptitle(
        f'Epoch {epoch} | Batch {batch_idx} | Mean Acc={mean_acc:.4f} | BCH={USE_BCH}',
        fontsize=10, fontweight='bold'
    )
    plt.tight_layout()
    fname = os.path.join(LOG_DIR, f'collage_ep{epoch:03d}_b{batch_idx:05d}.png')
    plt.savefig(fname, dpi=80, bbox_inches='tight')
    plt.close(fig)
    print(f'  🖼️  Collage → {os.path.basename(fname)}')


print('Collage logger ready ✅')


Collage logger ready ✅


## 🏋️ Cell 17 — Training Loop

In [ ]:
# ── Warm restart — run this once after updating lambdas ───────────
WARM_LR = 3e-4   # 3x original, gives the model a kick

for pg in opt_enc_dec.param_groups:
    pg['lr'] = WARM_LR
for pg in opt_disc.param_groups:
    pg['lr'] = WARM_LR * 0.5

# fresh cosine cycle from here
sched_enc_dec = optim.lr_scheduler.CosineAnnealingLR(
    opt_enc_dec, T_max=30, eta_min=LR_MIN
)
sched_disc = optim.lr_scheduler.CosineAnnealingLR(
    opt_disc, T_max=30, eta_min=LR_MIN
)

# verify
print('LR warm restart applied')
for pg in opt_enc_dec.param_groups:
    print(f'  enc_dec lr = {pg["lr"]}')
for pg in opt_disc.param_groups:
    print(f'  disc    lr = {pg["lr"]}')

LR warm restart applied
  enc_dec lr = 0.0003
  disc    lr = 0.00015


In [ ]:
# Paste this after Cell 9 to verify gradients will flow through Swin
for name, p in encoder.swin.named_parameters():
    p.requires_grad = True   # unfreeze Swin if timm froze it by default
print("Swin unfrozen ✅")

Swin unfrozen ✅


In [ ]:
def progress_bar(current, total, width=28):
    filled = int(width * current / max(total,1))
    return f'[{"█"*filled}{"░"*(width-filled)}] {current}/{total}'


print('=' * 65)
print(f'  Epochs={EPOCHS}  IMG={IMG_SIZE}x{IMG_SIZE}  MSG={MESSAGE_LENGTH}b  BCH={USE_BCH}')
print('=' * 65)

global_step = 0

for epoch in range(start_epoch, EPOCHS):
    encoder.train(); decoder.train(); discriminator.train(); noise_layer.train()

    ep_msg = ep_img = ep_acc = ep_psnr = 0.0
    nb = 0
    t0 = time.time()

    for batch_idx, cover in enumerate(train_loader):
        cover = cover.to(device, non_blocking=True)
        B     = cover.size(0)

        # ── 1. Message ────────────────────────────────────────────
        raw_msg = torch.randint(0, 2, (B, DATA_BITS), device=device).float()
        msg = bch_encode_bits(raw_msg) if USE_BCH else raw_msg

        # ── 2. Encoder + Decoder ──────────────────────────────────
        opt_enc_dec.zero_grad()
        encoded = encoder(cover, msg)
        noised  = noise_layer(encoded)
        decoded = decoder(noised)

        # ── 3. Generator loss ─────────────────────────────────────
        loss_msg  = bce_loss(decoded, msg)
        loss_img  = mse_loss(encoded, cover) + ssim_loss(encoded, cover)
        loss_freq = freq_loss(cover, encoded)
        disc_fake = discriminator(encoded)
        loss_adv  = adv_bce(disc_fake, torch.zeros_like(disc_fake))

        total_g = (LAMBDA_MSG*loss_msg + LAMBDA_IMG*loss_img
                   + LAMBDA_FREQ*loss_freq + LAMBDA_ADV*loss_adv)
        total_g.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(decoder.parameters()), 1.0)
        opt_enc_dec.step()

        # ── 4. Discriminator ──────────────────────────────────────
        opt_disc.zero_grad()
        loss_d = 0.5*(adv_bce(discriminator(cover), torch.zeros_like(discriminator(cover))) +
                      adv_bce(discriminator(encoded.detach()), torch.ones_like(disc_fake)))
        loss_d.backward()
        opt_disc.step()

        # ── 5. Metrics ────────────────────────────────────────────
        with torch.no_grad():
            acc_b  = bit_accuracy(decoded, msg)
            psnr_b = psnr(cover, encoded)
        ep_msg += loss_msg.item(); ep_img += loss_img.item()
        ep_acc += acc_b; ep_psnr += psnr_b
        nb += 1; global_step += 1

        # ── 6. Colab console output ────────────────────────────────
        if batch_idx % PRINT_EVERY == 0:
            bar = progress_bar(batch_idx+1, len(train_loader))
            print(f'Ep {epoch:02d}/{EPOCHS-1} {bar}  '
                  f'MsgL={loss_msg.item():.4f}  '
                  f'ImgL={loss_img.item():.4f}  '
                  f'Acc={acc_b*100:.1f}%  '
                  f'PSNR={psnr_b:.1f}dB  '
                  f't={time.time()-t0:.0f}s')

        # ── 7. Save collage to Drive ───────────────────────────────
        if global_step % LOG_EVERY_N_BATCHES == 0:
            encoder.eval(); decoder.eval()
            with torch.no_grad():
                enc_log = encoder(cover[:4], msg[:4])
                dec_log = decoder(enc_log)
            save_collage(cover[:4], enc_log, msg[:4], dec_log,
                         epoch, batch_idx, acc_b)
            encoder.train(); decoder.train()

    # ── End of epoch ─────────────────────────────────────────────────
    sched_enc_dec.step(); sched_disc.step()

    avg_msg  = ep_msg / nb
    avg_img  = ep_img / nb
    avg_acc  = ep_acc / nb
    avg_psnr = ep_psnr / nb

    history['epoch'].append(epoch)
    history['msg_loss'].append(avg_msg)
    history['img_loss'].append(avg_img)
    history['acc'].append(avg_acc)
    history['psnr'].append(avg_psnr)

    print('─' * 65)
    print(f'📊 Epoch {epoch:02d}  '
          f'MsgL={avg_msg:.4f}  ImgL={avg_img:.4f}  '
          f'Acc={avg_acc*100:.2f}%  PSNR={avg_psnr:.2f}dB')

    # Always overwrite last checkpoint (1 file only)
    save_checkpoint(LAST_CKPT, epoch, is_best=False)

    # Overwrite best checkpoint only when accuracy improves (1 file only)
    if avg_acc > best_acc:
        best_acc = avg_acc
        save_checkpoint(BEST_CKPT, epoch, is_best=True)
        print(f'  🎯 New best: {best_acc*100:.2f}%')
    print('─' * 65)

print('\n✅ Training complete!')
print(f'Best bit accuracy achieved: {best_acc*100:.2f}%')


  Epochs=100  IMG=128x128  MSG=64b  BCH=False
Ep 23/99 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 1/1000  MsgL=0.2502  ImgL=0.0085  Acc=86.4%  PSNR=41.8dB  t=7s
Ep 23/99 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 21/1000  MsgL=0.2519  ImgL=0.0092  Acc=85.2%  PSNR=43.2dB  t=31s
Ep 23/99 [█░░░░░░░░░░░░░░░░░░░░░░░░░░░] 41/1000  MsgL=0.2781  ImgL=0.0088  Acc=84.7%  PSNR=43.1dB  t=58s
Ep 23/99 [█░░░░░░░░░░░░░░░░░░░░░░░░░░░] 61/1000  MsgL=0.2544  ImgL=0.0065  Acc=84.8%  PSNR=42.1dB  t=85s
Ep 23/99 [██░░░░░░░░░░░░░░░░░░░░░░░░░░] 81/1000  MsgL=0.2721  ImgL=0.0079  Acc=85.8%  PSNR=41.9dB  t=113s
Ep 23/99 [██░░░░░░░░░░░░░░░░░░░░░░░░░░] 101/1000  MsgL=0.2596  ImgL=0.0085  Acc=85.7%  PSNR=42.2dB  t=138s
Ep 23/99 [███░░░░░░░░░░░░░░░░░░░░░░░░░] 121/1000  MsgL=0.2237  ImgL=0.0053  Acc=86.8%  PSNR=42.5dB  t=165s
Ep 23/99 [███░░░░░░░░░░░░░░░░░░░░░░░░░] 141/1000  MsgL=0.2562  ImgL=0.0101  Acc=85.4%  PSNR=43.1dB  t=193s
Ep 23/99 [████░░░░░░░░░░░░░░░░░░░░░░░░] 161/1000  MsgL=0.2371  ImgL=0.0056  Acc=86.5%  PSNR=42.9dB  t=220s
Ep

## 📈 Cell 18 — Training History Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))

axes[0].plot(history['epoch'], history['msg_loss'], 'b-o', ms=3)
axes[0].set_title('Message Loss (BCE)'); axes[0].set_xlabel('Epoch')
axes[0].grid(True, alpha=0.4)

axes[1].plot(history['epoch'], [a*100 for a in history['acc']], 'g-o', ms=3)
axes[1].axhline(100, color='r', ls='--', alpha=0.4, label='100%')
axes[1].set_title('Bit Accuracy (%)'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.4)

axes[2].plot(history['epoch'], history['psnr'], 'm-o', ms=3)
axes[2].axhline(40, color='g', ls='--', alpha=0.4, label='40dB')
axes[2].set_title('PSNR (dB)'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(True, alpha=0.4)

plt.suptitle('Swin Watermark — Training History', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'training_history.png'), dpi=100)
plt.show()
print('Plot saved to Drive ✅')


Plot saved to Drive ✅


## 🔬 Cell 19 — Inference Demo

In [91]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 1 — Patch & Sliding-Window Configuration
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Patch / model size (must match encoder/decoder training size) ──
PATCH_SIZE = 128          # encoder/decoder native resolution

# ── Sliding window steps ──────────────────────────────────────────
STEP_HORIZONTAL = 64      # column step during horizontal scan (left→right)
STEP_VERTICAL   = 64      # row step during vertical scan (top→down)

# ── Confidence weights ────────────────────────────────────────────
# CONF_MODE: "flat"    → aligned and non-aligned get fixed weights
#            "dynamic" → misaligned weight decays with distance from nearest grid point
CONF_MODE              = "flat"    # "flat" | "dynamic"
CONF_ALIGNED           = 1       # weight for perfectly grid-aligned windows
CONF_MISALIGNED_FLAT   = 1       # weight for all non-aligned windows (flat mode)

CONF_DYNAMIC_MIN       = 0.1       # minimum weight in dynamic mode
CONF_DYNAMIC_MAX       = 0.9       # maximum weight in dynamic mode (just below aligned)

# ── Scan aggregation mode ─────────────────────────────────────────
# AGGREGATION_MODE: "combined"     → pool all scan results into one vote
#                  "intermediate"  → each scan votes independently, then results are compared
AGGREGATION_MODE = "combined"      # "combined" | "intermediate"

# ── Output directory (Colab session storage) ──────────────────────
OUTPUT_BASE_DIR = "/content/watermark_outputs"


FEATHER_RADIUS = 0   # pixels — blend zone width at each patch boundary (try 8–20)
HISTOGRAM_MATCH = False


In [92]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 2 — Session Logger  (updated: CSV comparison output)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import os, json, csv, time
import numpy as np
from PIL import Image as PILImage
from datetime import datetime


class WatermarkLogger:
    def __init__(self, base_dir: str, run_tag: str = ""):
        ts  = datetime.now().strftime("%Y%m%d_%H%M%S")
        tag = f"_{run_tag}" if run_tag else ""
        self.run_dir = os.path.join(base_dir, f"run_{ts}{tag}")

        self.dirs = {
            "original":           os.path.join(self.run_dir, "original"),
            "patches":            os.path.join(self.run_dir, "patches"),
            "encoded_patches":    os.path.join(self.run_dir, "encoded_patches"),
            "stitched":           os.path.join(self.run_dir, "stitched"),
            "windows_horizontal": os.path.join(self.run_dir, "windows_horizontal"),
            "windows_vertical":   os.path.join(self.run_dir, "windows_vertical"),
        }
        for d in self.dirs.values():
            os.makedirs(d, exist_ok=True)

        self._window_log = []
        self._patch_log  = []
        self._t0         = time.time()

        print(f"[Logger] Run directory: {self.run_dir}")

    # ── internal helpers ──────────────────────────────────────────

    def _tensor_to_pil(self, tensor):
        arr = tensor.detach().cpu().permute(1, 2, 0).numpy()
        arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
        return PILImage.fromarray(arr)

    def _save_image(self, tensor_or_pil, folder_key: str, filename: str) -> str:
        path = os.path.join(self.dirs[folder_key], filename)
        if isinstance(tensor_or_pil, PILImage.Image):
            tensor_or_pil.save(path)
        else:
            self._tensor_to_pil(tensor_or_pil).save(path)
        return path

    def _write_json(self, data, filename: str) -> str:
        path = os.path.join(self.run_dir, filename)
        with open(path, "w") as f:
            json.dump(data, f, indent=2,
                      default=lambda x: x.tolist() if hasattr(x, "tolist") else str(x))
        return path

    def _write_csv(self, rows: list[dict], filename: str) -> str:
        path = os.path.join(self.run_dir, filename)
        if not rows:
            return path
        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        return path

    # ── public API ────────────────────────────────────────────────

    def log_config(self, config: dict):
        meta = {"timestamp": datetime.now().isoformat(), "run_dir": self.run_dir, "config": config}
        self._write_json(meta, "run_meta.json")
        print(f"[Logger] Config saved → run_meta.json")

    def log_original(self, image_tensor, image_id: str = "input"):
        path = self._save_image(image_tensor, "original", f"{image_id}.png")
        h, w = image_tensor.shape[-2], image_tensor.shape[-1]
        print(f"[Logger] Original image saved  ({w}×{h}) → {os.path.basename(path)}")

    def log_patch(self, patch_tensor, row: int, col: int, stage: str = "pre"):
        folder = "patches" if stage == "pre" else "encoded_patches"
        fname  = f"patch_r{row:02d}_c{col:02d}_{stage}.png"
        path   = self._save_image(patch_tensor, folder, fname)
        self._patch_log.append({
            "row": row, "col": col, "stage": stage,
            "pixel_y": row * PATCH_SIZE, "pixel_x": col * PATCH_SIZE,
            "path": path,
        })

    def log_stitched(self, image_tensor, label: str = "watermarked"):
        path = self._save_image(image_tensor, "stitched", f"{label}.png")
        h, w = image_tensor.shape[-2], image_tensor.shape[-1]
        print(f"[Logger] Stitched image saved  ({w}×{h}) → {os.path.basename(path)}")

    def log_window(self, window_tensor, scan_type, win_x, win_y,
                   raw_probs, confidence, is_aligned, window_index):
        folder = "windows_horizontal" if scan_type == "horizontal" else "windows_vertical"
        fname  = f"win_{scan_type[0]}{window_index:04d}_x{win_x:04d}_y{win_y:04d}.png"
        path   = self._save_image(window_tensor, folder, fname)
        bits   = (raw_probs > 0.5).astype(np.uint8)
        self._window_log.append({
            "index":      window_index,
            "scan_type":  scan_type,
            "x":          win_x,
            "y":          win_y,
            "is_aligned": is_aligned,
            "confidence": round(confidence, 6),
            "bits":       bits.tolist(),
            "probs":      np.round(raw_probs, 5).tolist(),
            "path":       path,
        })

    def log_final_result(self, final_bits: np.ndarray, summary: dict):
        result = {
            "final_bits":      final_bits.tolist(),
            "final_bits_hex":  _bits_to_hex_np(final_bits),
            "total_windows":   len(self._window_log),
            "elapsed_seconds": round(time.time() - self._t0, 2),
            **summary,
        }
        self._write_json(result, "final_result.json")
        print(f"[Logger] Final result saved → final_result.json")
        print(f"         Bits (hex): {result['final_bits_hex']}")
        print(f"         Windows processed: {result['total_windows']}")
        print(f"         Elapsed: {result['elapsed_seconds']}s")
        return result

    def flush_window_log(self):
        # ── JSON (full detail, probs + paths) ─────────────────────
        self._write_json(self._window_log, "window_results.json")

        # ── CSV (flat, human-readable, no nested lists) ────────────
        rows = []
        for w in self._window_log:
            row = {
                "index":      w["index"],
                "scan_type":  w["scan_type"],
                "x":          w["x"],
                "y":          w["y"],
                "is_aligned": int(w["is_aligned"]),
                "confidence": w["confidence"],
            }
            for i, (b, p) in enumerate(zip(w["bits"], w["probs"])):
                row[f"bit_{i:03d}"]  = b
                row[f"prob_{i:03d}"] = round(p, 5)
            rows.append(row)

        path = self._write_csv(rows, "window_results.csv")
        print(f"[Logger] Window log flushed → window_results.csv  ({len(rows)} rows)")

    def flush_patch_log(self):
        self._write_json(self._patch_log, "patch_log.json")

    # ── NEW: bits comparison CSV ───────────────────────────────────

    def save_bits_comparison_csv(
        self,
        original_bits: np.ndarray,        # (MESSAGE_LENGTH,)  ground-truth
        final_bits:    np.ndarray,         # (MESSAGE_LENGTH,)  voted result
    ):
        """
        Builds a per-bit comparison table:

        Columns
        -------
        bit_index         : 0 … MESSAGE_LENGTH-1
        original          : ground-truth bit
        [one col per aligned window]  aligned_r<R>_c<C>_<scan>  : bit from that window
        final_vote        : final decided bit
        original_vs_final : 1 if match, 0 if flip
        aligned_consensus : fraction of aligned windows that agree with final_vote
        """

        msg_len = len(original_bits)

        # ── collect aligned windows only ──────────────────────────
        aligned = [w for w in self._window_log if w["is_aligned"]]
        if not aligned:
            print("[Logger] No aligned windows found — skipping bits_comparison.csv")
            return

        # ── build column headers for aligned windows ──────────────
        aligned_cols = []
        for w in aligned:
            grid_r = w["y"] // PATCH_SIZE
            grid_c = w["x"] // PATCH_SIZE
            col    = f"aligned_r{grid_r:02d}_c{grid_c:02d}_{w['scan_type'][0]}"
            aligned_cols.append((col, w["bits"]))

        # ── one row per bit ───────────────────────────────────────
        rows = []
        for i in range(msg_len):
            orig  = int(original_bits[i])
            final = int(final_bits[i])

            row = {"bit_index": i, "original": orig}

            # aligned window bits for this bit position
            aligned_bit_values = []
            for col_name, bits in aligned_cols:
                b = int(bits[i]) if i < len(bits) else -1
                row[col_name] = b
                aligned_bit_values.append(b)

            row["final_vote"]        = final
            row["original_vs_final"] = int(orig == final)

            # what fraction of aligned windows agree with final_vote
            valid = [b for b in aligned_bit_values if b >= 0]
            row["aligned_consensus"] = (
                round(sum(1 for b in valid if b == final) / len(valid), 4)
                if valid else 0.0
            )

            rows.append(row)

        path = self._write_csv(rows, "bits_comparison.csv")

        # ── summary print ─────────────────────────────────────────
        total      = len(rows)
        correct    = sum(r["original_vs_final"] for r in rows)
        bit_acc    = correct / total if total else 0.0
        print(f"[Logger] Bits comparison saved → bits_comparison.csv")
        print(f"         Bit accuracy (original vs final vote): {bit_acc*100:.2f}%  ({correct}/{total})")


# ── hex helper ────────────────────────────────────────────────────
def _bits_to_hex_np(bits: np.ndarray) -> str:
    bits = bits.astype(np.uint8).flatten()
    pad  = (8 - len(bits) % 8) % 8
    bits = np.concatenate([bits, np.zeros(pad, dtype=np.uint8)])
    return "0x" + "".join(f"{b:02x}" for b in np.packbits(bits))

In [93]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 3 — Patch Encoder  (updated: feathered stitching)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import torch
import torch.nn.functional as F

def _histogram_match_patch(
    encoded: torch.Tensor,   # (1, 3, H, W)  encoded patch
    original: torch.Tensor,  # (1, 3, H, W)  original patch
) -> torch.Tensor:
    """
    For each channel independently, rescale the encoded patch so its
    mean and std match the original patch.

    This removes patch-level brightness/contrast drift introduced by
    the encoder while keeping the high-frequency watermark signal intact.
    """
    out = encoded.clone()
    for ch in range(encoded.shape[1]):
        enc_ch  = encoded[:, ch, :, :]
        ori_ch  = original[:, ch, :, :]

        enc_mean, enc_std = enc_ch.mean(), enc_ch.std().clamp(min=1e-6)
        ori_mean, ori_std = ori_ch.mean(), ori_ch.std().clamp(min=1e-6)

        # shift encoded channel to match original mean+std
        normalized       = (enc_ch - enc_mean) / enc_std
        out[:, ch, :, :] = normalized * ori_std + ori_mean

    return out.clamp(0, 1)


def _build_feather_mask(rows: int, cols: int, patch: int, radius: int) -> torch.Tensor:
    """
    Builds a (1, 1, H, W) weight mask where values smoothly ramp
    from 0→1 near every interior patch boundary using a cosine fade.
    Pixels far from any boundary = 1.0 (fully keep encoded value).
    Pixels at the boundary edge = 0.0 (fully keep original value).
    """
    H, W = rows * patch, cols * patch
    mask = torch.ones(1, 1, H, W, dtype=torch.float32)

    # cosine ramp: 0 at boundary, 1 at radius distance away
    def cosine_ramp(n: int) -> torch.Tensor:
        t = torch.arange(n, dtype=torch.float32) / (n - 1)
        return 0.5 - 0.5 * torch.cos(t * torch.pi)   # 0 → 1

    ramp = cosine_ramp(radius)     # length=radius, goes 0→1
    fade = torch.cat([ramp, torch.ones(patch - 2 * radius), ramp.flip(0)])
    # fade has length = patch, dips to 0 at both ends

    # apply per-column boundary fade (vertical seams)
    for c in range(1, cols):
        x0 = c * patch
        x_start = max(0, x0 - radius)
        x_end   = min(W, x0 + radius)
        zone_w  = x_end - x_start
        col_ramp = torch.cat([ramp[-(x0 - x_start):], ramp[:x_end - x0]])[:zone_w]
        mask[:, :, :, x_start:x_end] *= col_ramp.view(1, 1, 1, zone_w)

    # apply per-row boundary fade (horizontal seams)
    for r in range(1, rows):
        y0 = r * patch
        y_start = max(0, y0 - radius)
        y_end   = min(H, y0 + radius)
        zone_h  = y_end - y_start
        row_ramp = torch.cat([ramp[-(y0 - y_start):], ramp[:y_end - y0]])[:zone_h]
        mask[:, :, y_start:y_end, :] *= row_ramp.view(1, 1, zone_h, 1)

    return mask   # (1, 1, H, W) values in [0, 1]


def encode_large_image(
    image_tensor: torch.Tensor,
    message_bits: torch.Tensor,
    encoder,
    logger,
    device,
) -> torch.Tensor:
    _, C, H, W = image_tensor.shape
    assert H % PATCH_SIZE == 0 and W % PATCH_SIZE == 0
    rows, cols = H // PATCH_SIZE, W // PATCH_SIZE
    print(f"[Encoder] Grid: {rows}×{cols} = {rows*cols} patches  |  image {W}×{H}")
    print(f"[Encoder] Histogram match : {HISTOGRAM_MATCH}")
    print(f"[Encoder] Feather radius  : {FEATHER_RADIUS}px")

    logger.log_original(image_tensor[0])

    encoded_canvas = torch.zeros_like(image_tensor)
    encoder.eval()

    with torch.no_grad():
        for r in range(rows):
            for c in range(cols):
                y0, y1 = r * PATCH_SIZE, (r + 1) * PATCH_SIZE
                x0, x1 = c * PATCH_SIZE, (c + 1) * PATCH_SIZE

                patch   = image_tensor[:, :, y0:y1, x0:x1].to(device)
                logger.log_patch(patch[0], row=r, col=c, stage="pre")

                encoded = encoder(patch, message_bits.to(device))

                # ── histogram match: remove brightness drift ──────
                if HISTOGRAM_MATCH:
                    encoded = _histogram_match_patch(encoded.cpu(), patch.cpu()).to(device)

                logger.log_patch(encoded[0], row=r, col=c, stage="post")
                encoded_canvas[:, :, y0:y1, x0:x1] = encoded.cpu()

    # ── feathered blend at patch boundaries ───────────────────────
    feather_mask = _build_feather_mask(rows, cols, PATCH_SIZE, FEATHER_RADIUS)
    watermarked  = feather_mask * encoded_canvas + (1.0 - feather_mask) * image_tensor

    logger.log_stitched(watermarked[0], label="watermarked")
    logger.flush_patch_log()
    print(f"[Encoder] Stitching complete ✅")
    return watermarked

In [94]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 4 — Confidence Calculator
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def compute_confidence(win_x: int, win_y: int) -> tuple[float, bool]:
    """
    Returns (confidence_weight, is_aligned) for a window at pixel (win_x, win_y).

    Aligned  = both x and y are exact multiples of PATCH_SIZE.
    The confidence mode is controlled by CONF_MODE global variable.
    """
    x_rem = win_x % PATCH_SIZE
    y_rem = win_y % PATCH_SIZE
    is_aligned = (x_rem == 0) and (y_rem == 0)

    if is_aligned:
        return CONF_ALIGNED, True

    if CONF_MODE == "flat":
        return CONF_MISALIGNED_FLAT, False

    # "dynamic" — linear decay based on distance to nearest grid point
    # distance in each axis: min of remainder and (PATCH_SIZE - remainder)
    dx = min(x_rem, PATCH_SIZE - x_rem)
    dy = min(y_rem, PATCH_SIZE - y_rem)
    max_dist = PATCH_SIZE / 2                           # max possible per axis
    norm_dist = ((dx ** 2 + dy ** 2) ** 0.5) / (max_dist * 2 ** 0.5)  # 0→1
    weight = CONF_DYNAMIC_MAX - norm_dist * (CONF_DYNAMIC_MAX - CONF_DYNAMIC_MIN)
    return round(weight, 6), False

In [95]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 5 — Sliding Window Scanner
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _run_scan(
    image_tensor: torch.Tensor,   # (1, 3, H, W)
    decoder,
    logger: WatermarkLogger,
    device,
    scan_type: str,               # "horizontal" | "vertical"
) -> list[dict]:
    """
    Internal: runs one full scan pass and returns list of window result dicts.

    Horizontal scan
    ---------------
      Rows are spaced exactly PATCH_SIZE apart  (y = 0, 128, 256, ...)
      Columns step by STEP_HORIZONTAL           (x = 0, 64, 128, ...)

    Vertical scan
    -------------
      Columns are spaced exactly PATCH_SIZE apart  (x = 0, 128, 256, ...)
      Rows step by STEP_VERTICAL                   (y = 0, 64, 128, ...)

    Any window that would exceed image bounds is skipped (no padding).
    """
    _, C, H, W = image_tensor.shape
    P = PATCH_SIZE
    results = []
    win_idx = 0
    decoder.eval()

    if scan_type == "horizontal":
        y_positions = range(0, H - P + 1, P)                # row stride = PATCH_SIZE
        x_positions = range(0, W - P + 1, STEP_HORIZONTAL)  # col stride = STEP_H
    else:  # vertical
        y_positions = range(0, H - P + 1, STEP_VERTICAL)    # row stride = STEP_V
        x_positions = range(0, W - P + 1, P)                # col stride = PATCH_SIZE

    total = len(list(y_positions)) * len(list(x_positions))
    print(f"[Scanner:{scan_type}] {total} windows to process ...")

    with torch.no_grad():
        for win_y in y_positions:
            for win_x in x_positions:
                crop = image_tensor[:, :, win_y:win_y+P, win_x:win_x+P].to(device)

                probs = decoder(crop)                    # (1, MESSAGE_LENGTH) sigmoid
                probs_np = probs[0].cpu().numpy()        # (MESSAGE_LENGTH,)

                confidence, is_aligned = compute_confidence(win_x, win_y)

                logger.log_window(
                    window_tensor = crop[0].cpu(),
                    scan_type     = scan_type,
                    win_x         = win_x,
                    win_y         = win_y,
                    raw_probs     = probs_np,
                    confidence    = confidence,
                    is_aligned    = is_aligned,
                    window_index  = win_idx,
                )

                results.append({
                        "index":      win_idx,        # ← add this line
                    "scan_type":  scan_type,
                    "x":          win_x,
                    "y":          win_y,
                    "probs":      probs_np,
                    "confidence": confidence,
                            "bits":       (probs_np > 0.5).astype(np.uint8),   # ← add this
                    "is_aligned": is_aligned,
                })
                win_idx += 1

    print(f"[Scanner:{scan_type}] Done — {win_idx} windows scanned ✅")
    return results


def scan_watermarked_image(
    image_tensor: torch.Tensor,
    decoder,
    logger: WatermarkLogger,
    device,
) -> tuple[list[dict], list[dict]]:
    """
    Runs both horizontal and vertical scans.
    Returns (horizontal_results, vertical_results).
    """
    h_results = _run_scan(image_tensor, decoder, logger, device, "horizontal")
    v_results = _run_scan(image_tensor, decoder, logger, device, "vertical")
    logger.flush_window_log()
    return h_results, v_results

In [96]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 6 — Weighted Voting & Final Bit Decision  (updated)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Vote pool control ─────────────────────────────────────────────
# "all"                   → all windows vote (original behaviour)
# "aligned_only"          → only perfectly grid-aligned windows vote
# "aligned_then_fallback" → aligned first; use all on near-tie bits
VOTE_POOL      = "all"
TIE_THRESHOLD  = 0.1   # used only when VOTE_POOL = "aligned_then_fallback"
VOTE_METHOD = "soft"



def _weighted_vote(window_results: list[dict]) -> tuple[np.ndarray, np.ndarray]:
    if not window_results:
        raise ValueError("No window results to vote on.")

    msg_len       = len(window_results[0]["probs"])
    weighted_ones = np.zeros(msg_len, dtype=np.float64)
    weight_total  = 0.0

    for w in window_results:
        c = w["confidence"]
        if VOTE_METHOD == "hard":
            # threshold first → clean 0/1 → weighted count
            bits = (np.array(w["probs"]) > 0.5).astype(np.float64)
            weighted_ones += c * bits
        else:
            # soft: use raw sigmoid probabilities directly
            weighted_ones += c * np.array(w["probs"])
        weight_total += c

    avg_probs  = weighted_ones / max(weight_total, 1e-9)
    final_bits = (avg_probs > 0.5).astype(np.uint8)
    return final_bits, avg_probs



def decide_final_bits(
    h_results:        list[dict],
    v_results:        list[dict],
    logger:           WatermarkLogger,
    original_message: np.ndarray | None = None,
) -> np.ndarray:
    """
    Aggregates horizontal + vertical scan results into a final bit array.

    VOTE_POOL controls which windows participate:
      "all"                   → every window
      "aligned_only"          → only grid-aligned windows
      "aligned_then_fallback" → aligned windows; fall back to all on tie bits

    AGGREGATION_MODE controls how H and V scans are combined:
      "combined"     → pool H + V together, one vote
      "intermediate" → H and V vote separately, average their soft scores
    """
    all_results     = h_results + v_results
    aligned_results = [w for w in all_results if w["is_aligned"]]

    print(f"[Voting] Total windows : {len(all_results)}")
    print(f"[Voting] Aligned       : {len(aligned_results)}")
    print(f"[Voting] Pool mode     : {VOTE_POOL}")
    print(f"[Voting] Aggregation   : {AGGREGATION_MODE}")

    # ── select pool ───────────────────────────────────────────────
    if VOTE_POOL == "aligned_only":
        pool = aligned_results if aligned_results else all_results

    elif VOTE_POOL == "aligned_then_fallback":
        if not aligned_results:
            pool = all_results
        else:
            a_bits, a_probs = _weighted_vote(aligned_results)
            f_bits, f_probs = _weighted_vote(all_results)
            is_tie          = np.abs(a_probs - 0.5) < TIE_THRESHOLD
            final_probs     = np.where(is_tie, f_probs, a_probs)
            final_bits      = (final_probs > 0.5).astype(np.uint8)

            summary = {
                "aggregation_mode": AGGREGATION_MODE,
                "vote_pool":        VOTE_POOL,
                "aligned_windows":  len(aligned_results),
                "total_windows":    len(all_results),
                "tie_bits_count":   int(is_tie.sum()),
                "avg_probs_mean":   float(np.mean(final_probs)),
                "avg_probs_std":    float(np.std(final_probs)),
            }
            if original_message is not None:
                orig = np.array(original_message).flatten().astype(np.uint8)
                acc  = float(np.mean(final_bits[:len(orig)] == orig[:len(final_bits)]))
                summary["bit_accuracy_vs_original"] = round(acc, 6)
                print(f"[Voting] Bit accuracy vs original: {acc*100:.2f}%")
            logger.log_final_result(final_bits, summary)
            return final_bits

    else:  # "all"
        pool = all_results

    # ── vote on selected pool ─────────────────────────────────────
    if AGGREGATION_MODE == "combined":
        final_bits, avg_probs = _weighted_vote(pool)
        summary = {
            "aggregation_mode": AGGREGATION_MODE,
            "vote_pool":        VOTE_POOL,
            "aligned_windows":  len(aligned_results),
            "total_windows":    len(all_results),
            "voted_on":         len(pool),
            "avg_probs_mean":   float(np.mean(avg_probs)),
            "avg_probs_std":    float(np.std(avg_probs)),
        }

    else:  # "intermediate"
        h_pool = [w for w in h_results if (w["is_aligned"] if VOTE_POOL == "aligned_only" else True)]
        v_pool = [w for w in v_results if (w["is_aligned"] if VOTE_POOL == "aligned_only" else True)]
        h_bits, h_probs = _weighted_vote(h_pool)
        v_bits, v_probs = _weighted_vote(v_pool)
        avg_probs   = (h_probs + v_probs) / 2.0
        final_bits  = (avg_probs > 0.5).astype(np.uint8)
        agreement   = float(np.mean(h_bits == v_bits))
        summary = {
            "aggregation_mode":  AGGREGATION_MODE,
            "vote_pool":         VOTE_POOL,
            "aligned_windows":   len(aligned_results),
            "h_windows_voted":   len(h_pool),
            "v_windows_voted":   len(v_pool),
            "h_v_bit_agreement": round(agreement, 4),
            "avg_probs_mean":    float(np.mean(avg_probs)),
            "avg_probs_std":     float(np.std(avg_probs)),
        }

    if original_message is not None:
        orig = np.array(original_message).flatten().astype(np.uint8)
        acc  = float(np.mean(final_bits[:len(orig)] == orig[:len(final_bits)]))
        summary["bit_accuracy_vs_original"] = round(acc, 6)
        print(f"[Voting] Bit accuracy vs original: {acc*100:.2f}%")

    logger.log_final_result(final_bits, summary)
    return final_bits

In [97]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 7 — Full Pipeline  (updated: disable_logging + return h/v results)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class _NullLogger:
    """Drop-in logger that does nothing — used during batch runs to skip disk I/O."""
    run_dir     = None
    _window_log = []
    _patch_log  = []

    def log_config(self, *a, **kw):              pass
    def log_original(self, *a, **kw):            pass
    def log_patch(self, *a, **kw):               pass
    def log_stitched(self, *a, **kw):            pass
    def log_window(self, *a, **kw):              pass
    def log_final_result(self, *a, **kw):        pass
    def flush_window_log(self, *a, **kw):        pass
    def flush_patch_log(self, *a, **kw):         pass
    def save_bits_comparison_csv(self, *a, **kw): pass


def run_watermark_pipeline(
    image_tensor:    torch.Tensor,
    message_bits:    torch.Tensor,
    encoder,
    decoder,
    device,
    run_tag:         str  = "",
    skip_encode:     bool = False,
    disable_logging: bool = False,
) -> tuple[torch.Tensor, np.ndarray, list, list]:
    """
    Returns
    -------
    watermarked  : torch.Tensor  (1, 3, H, W)
    final_bits   : np.ndarray    (MESSAGE_LENGTH,)
    h_results    : list[dict]    per-window horizontal scan results
    v_results    : list[dict]    per-window vertical scan results
    """

    logger = _NullLogger() if disable_logging else WatermarkLogger(
        base_dir=OUTPUT_BASE_DIR, run_tag=run_tag
    )

    config_snapshot = {
        "PATCH_SIZE":            PATCH_SIZE,
        "STEP_HORIZONTAL":       STEP_HORIZONTAL,
        "STEP_VERTICAL":         STEP_VERTICAL,
        "CONF_MODE":             CONF_MODE,
        "CONF_ALIGNED":          CONF_ALIGNED,
        "CONF_MISALIGNED_FLAT":  CONF_MISALIGNED_FLAT,
        "CONF_DYNAMIC_MIN":      CONF_DYNAMIC_MIN,
        "CONF_DYNAMIC_MAX":      CONF_DYNAMIC_MAX,
        "AGGREGATION_MODE":      AGGREGATION_MODE,
        "VOTE_POOL":             VOTE_POOL,
        "VOTE_METHOD":           VOTE_METHOD,
        "skip_encode":           skip_encode,
        "disable_logging":       disable_logging,
        "image_shape":           list(image_tensor.shape),
        "message_bits_hex":      _bits_to_hex_np(message_bits[0].cpu().numpy()),
    }
    logger.log_config(config_snapshot)

    # ── Phase 1: Encode ───────────────────────────────────────────
    if skip_encode:
        watermarked = image_tensor
        logger.log_stitched(watermarked[0], label="provided_watermarked")
    else:
        watermarked = encode_large_image(image_tensor, message_bits, encoder, logger, device)

    # ── Phase 2: Sliding-window decode ────────────────────────────
    h_results, v_results = scan_watermarked_image(watermarked, decoder, logger, device)

    # ── Phase 3: Vote ─────────────────────────────────────────────
    original_np = message_bits[0].cpu().numpy().astype(np.uint8)
    final_bits  = decide_final_bits(h_results, v_results, logger, original_message=original_np)

    # ── Save bits comparison CSV (skipped silently if NullLogger) ──
    logger.save_bits_comparison_csv(
        original_bits = original_np,
        final_bits    = final_bits,
    )

    return watermarked, final_bits, h_results, v_results

In [98]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 8 — Batch Evaluation  (10 images × 10 runs each)
#           Logger saves only the single best + single worst run
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import requests, io
import torchvision.transforms as T
from PIL import Image as PILImage

# ── Config ────────────────────────────────────────────────────────
TARGET_SIZE  = 512
NUM_IMAGES   = 10
RUNS_PER_IMG = 10
IMAGE_SEEDS  = list(range(1, NUM_IMAGES + 1))

# ── Fetch helper ──────────────────────────────────────────────────
def fetch_image(seed: int) -> torch.Tensor:
    url      = f"https://picsum.photos/{TARGET_SIZE}/{TARGET_SIZE}?random={seed}"
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    pil_img  = PILImage.open(io.BytesIO(response.content)).convert("RGB")
    return T.ToTensor()(pil_img).unsqueeze(0)

# ── Buffer: store everything in memory, log nothing yet ───────────
# Each entry: {seed, run, acc, image_tensor, message_bits, watermarked, final_bits, orig_np}
all_results  = {seed: [] for seed in IMAGE_SEEDS}
run_buffer   = []   # flat list of all 100 run dicts

# ── Run (silent pipeline — suppress logger writes) ────────────────
print(f"\nRunning {NUM_IMAGES} images × {RUNS_PER_IMG} runs each ...\n")

for img_idx, seed in enumerate(IMAGE_SEEDS):

    print(f"{'─'*55}")
    print(f"  Image {img_idx+1}/{NUM_IMAGES}  (picsum seed={seed})")
    print(f"{'─'*55}")

    image_tensor = fetch_image(seed)

    for run in range(RUNS_PER_IMG):

        raw_msg      = torch.randint(0, 2, (1, DATA_BITS), device=device).float()
        message_bits = bch_encode_bits(raw_msg) if USE_BCH else raw_msg
        orig_np      = message_bits[0].cpu().numpy().astype(np.uint8)

        # run pipeline with logging fully disabled
        watermarked, final_bits, h_results, v_results = run_watermark_pipeline(
            image_tensor = image_tensor,
            message_bits = message_bits,
            encoder      = encoder,
            decoder      = decoder,
            device       = device,
            run_tag      = f"img{seed}_run{run+1}",
            disable_logging = True,   # ← new flag, see Block 7 update below
        )

        acc = float(np.mean(final_bits[:len(orig_np)] == orig_np[:len(final_bits)]))
        all_results[seed].append(acc)

        entry = {
            "seed":         seed,
            "run":          run + 1,
            "acc":          acc,
            "image_tensor": image_tensor,
            "message_bits": message_bits,
            "orig_np":      orig_np,
            "watermarked":  watermarked,
            "final_bits":   final_bits,
            "h_results":    h_results,
            "v_results":    v_results,
        }
        run_buffer.append(entry)

        print(f"    run {run+1:02d}/{RUNS_PER_IMG}  "
              f"orig={_bits_to_hex_np(orig_np)}  "
              f"recv={_bits_to_hex_np(final_bits)}  "
              f"acc={acc*100:.2f}%")

# ── Find global best and worst across all 100 runs ────────────────
best_entry = max(run_buffer, key=lambda e: e["acc"])
worst_entry = min(run_buffer, key=lambda e: e["acc"])

print(f"\n[Batch] Best  run: img{best_entry['seed']} run{best_entry['run']:02d}  acc={best_entry['acc']*100:.2f}%")
print(f"[Batch] Worst run: img{worst_entry['seed']} run{worst_entry['run']:02d}  acc={worst_entry['acc']*100:.2f}%")

# ── Replay logging for best and worst only ────────────────────────
for label, entry in [("BEST", best_entry), ("WORST", worst_entry)]:

    print(f"\n[Batch] Saving {label} run to logger ...")

    logger = WatermarkLogger(
        base_dir = OUTPUT_BASE_DIR,
        run_tag  = f"batch_{label.lower()}_img{entry['seed']}_run{entry['run']:02d}",
    )

    config_snapshot = {
        "batch_label":           label,
        "seed":                  entry["seed"],
        "run":                   entry["run"],
        "bit_accuracy":          round(entry["acc"], 6),
        "PATCH_SIZE":            PATCH_SIZE,
        "STEP_HORIZONTAL":       STEP_HORIZONTAL,
        "STEP_VERTICAL":         STEP_VERTICAL,
        "CONF_MODE":             CONF_MODE,
        "CONF_ALIGNED":          CONF_ALIGNED,
        "CONF_MISALIGNED_FLAT":  CONF_MISALIGNED_FLAT,
        "VOTE_POOL":             VOTE_POOL,
        "VOTE_METHOD":           VOTE_METHOD,
        "AGGREGATION_MODE":      AGGREGATION_MODE,
        "message_bits_hex":      _bits_to_hex_np(entry["orig_np"]),
        "final_bits_hex":        _bits_to_hex_np(entry["final_bits"]),
    }
    logger.log_config(config_snapshot)
    logger.log_original(entry["image_tensor"][0])
    logger.log_stitched(entry["watermarked"][0], label="watermarked")

    # re-register window results into logger so CSV and comparison save correctly
    for w in entry["h_results"] + entry["v_results"]:
        logger._window_log.append(w)

    logger.flush_window_log()

    summary = {
        "aggregation_mode":      AGGREGATION_MODE,
        "vote_pool":             VOTE_POOL,
        "aligned_windows":       sum(1 for w in logger._window_log if w["is_aligned"]),
        "total_windows":         len(logger._window_log),
        "bit_accuracy_vs_original": round(entry["acc"], 6),
    }
    logger.log_final_result(entry["final_bits"], summary)

    logger.save_bits_comparison_csv(
        original_bits = entry["orig_np"],
        final_bits    = entry["final_bits"],
    )

# ── Summary table ─────────────────────────────────────────────────
per_run_header = [f"r{r+1:02d}" for r in range(RUNS_PER_IMG)]
run_col_w      = 7
full_col_w     = [6] + [run_col_w] * RUNS_PER_IMG + [8, 7, 7]

def _full_row(*cells):
    widths = full_col_w + [8] * max(0, len(cells) - len(full_col_w))
    return "│ " + " │ ".join(str(c).rjust(w) for c, w in zip(cells, widths)) + " │"

def _full_divider(char="─"):
    return "├" + "┼".join(char * (w + 2) for w in full_col_w) + "┤"

full_top    = "┌" + "┬".join("─" * (w + 2) for w in full_col_w) + "┐"
full_header = "╞" + "╪".join("═" * (w + 2) for w in full_col_w) + "╡"
full_foot   = "└" + "┴".join("─" * (w + 2) for w in full_col_w) + "┘"

print(f"\n\n{'━'*80}")
print(f"  BATCH EVALUATION RESULTS  —  {NUM_IMAGES} images × {RUNS_PER_IMG} runs")
print(f"  VOTE_POOL={VOTE_POOL}  VOTE_METHOD={VOTE_METHOD}  AGGREGATION={AGGREGATION_MODE}")
print(f"{'━'*80}\n")

print(full_top)
print(_full_row("img", *per_run_header, "mean%", "min%", "max%"))
print(full_header)

grand_accs = []
for img_idx, seed in enumerate(IMAGE_SEEDS):
    accs      = all_results[seed]
    mean_acc  = np.mean(accs) * 100
    min_acc   = np.min(accs)  * 100
    max_acc   = np.max(accs)  * 100
    grand_accs.extend(accs)

    run_cells = [f"{a*100:.1f}" for a in accs]
    print(_full_row(f"img{seed}", *run_cells, f"{mean_acc:.2f}", f"{min_acc:.1f}", f"{max_acc:.1f}"))
    if img_idx < NUM_IMAGES - 1:
        print(_full_divider())

print(full_foot)

grand_mean = np.mean(grand_accs) * 100
grand_min  = np.min(grand_accs)  * 100
grand_max  = np.max(grand_accs)  * 100
grand_std  = np.std(grand_accs)  * 100
perfect    = sum(1 for a in grand_accs if a == 1.0)
total_runs = NUM_IMAGES * RUNS_PER_IMG

print(f"\n{'─'*55}")
print(f"  Grand mean accuracy : {grand_mean:.2f}%")
print(f"  Std deviation       : {grand_std:.2f}%")
print(f"  Min / Max           : {grand_min:.2f}% / {grand_max:.2f}%")
print(f"  Perfect runs (100%) : {perfect}/{total_runs}")
print(f"  Logger saved        : BEST (img{best_entry['seed']} r{best_entry['run']:02d}) + "
      f"WORST (img{worst_entry['seed']} r{worst_entry['run']:02d})")
print(f"{'─'*55}\n")


Running 10 images × 10 runs each ...

───────────────────────────────────────────────────────
  Image 1/10  (picsum seed=1)
───────────────────────────────────────────────────────
[Encoder] Grid: 4×4 = 16 patches  |  image 512×512
[Encoder] Histogram match : False
[Encoder] Feather radius  : 0px
[Encoder] Stitching complete ✅
[Scanner:horizontal] 28 windows to process ...
[Scanner:horizontal] Done — 28 windows scanned ✅
[Scanner:vertical] 28 windows to process ...
[Scanner:vertical] Done — 28 windows scanned ✅
[Voting] Total windows : 56
[Voting] Aligned       : 32
[Voting] Pool mode     : all
[Voting] Aggregation   : combined
[Voting] Bit accuracy vs original: 90.62%
    run 01/10  orig=0x26a40344f2bf5e66  recv=0x26940b44e2bd5e62  acc=90.62%
[Encoder] Grid: 4×4 = 16 patches  |  image 512×512
[Encoder] Histogram match : False
[Encoder] Feather radius  : 0px
[Encoder] Stitching complete ✅
[Scanner:horizontal] 28 windows to process ...
[Scanner:horizontal] Done — 28 windows scanned ✅
[S

KeyboardInterrupt: 

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# DIAGNOSTIC — visualize stitching artifact
# Run after a single run_watermark_pipeline() call
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def diagnose_stitch_artifact(image_tensor, watermarked, save_path=None):
    """
    Shows 5 panels:
      1. Original
      2. Watermarked (stitched)
      3. Difference × 20  (amplified to make artifact visible)
      4. Difference × 20  with grid overlay showing patch boundaries
      5. Per-patch mean brightness shift (heatmap)
    """
    orig = image_tensor[0].cpu().permute(1,2,0).numpy()      # HWC
    wm   = watermarked[0].detach().cpu().permute(1,2,0).numpy()
    diff = wm - orig                                          # signed diff

    H, W, _ = orig.shape
    rows, cols = H // PATCH_SIZE, W // PATCH_SIZE

    # per-patch mean absolute diff
    patch_diff = np.zeros((rows, cols))
    patch_mean_shift = np.zeros((rows, cols, 3))
    for r in range(rows):
        for c in range(cols):
            y0, y1 = r*PATCH_SIZE, (r+1)*PATCH_SIZE
            x0, x1 = c*PATCH_SIZE, (c+1)*PATCH_SIZE
            pd = diff[y0:y1, x0:x1, :]
            patch_diff[r, c]       = np.abs(pd).mean()
            patch_mean_shift[r, c] = pd.mean(axis=(0,1))   # per-channel mean shift

    fig = plt.figure(figsize=(20, 10))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.25)

    # ── Panel 1: original ─────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(np.clip(orig, 0, 1))
    ax1.set_title("Original", fontsize=11)
    ax1.axis("off")

    # ── Panel 2: watermarked ──────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(np.clip(wm, 0, 1))
    ax2.set_title("Watermarked (stitched)", fontsize=11)
    ax2.axis("off")
    # draw patch grid on watermarked
    for r in range(1, rows):
        ax2.axhline(r * PATCH_SIZE - 0.5, color="red", lw=0.8, alpha=0.6)
    for c in range(1, cols):
        ax2.axvline(c * PATCH_SIZE - 0.5, color="red", lw=0.8, alpha=0.6)

    # ── Panel 3: diff × 20 ────────────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    diff_vis = np.clip(diff * 20 + 0.5, 0, 1)
    ax3.imshow(diff_vis)
    ax3.set_title("Difference × 20", fontsize=11)
    ax3.axis("off")

    # ── Panel 4: diff × 20 + grid ────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.imshow(diff_vis)
    for r in range(1, rows):
        ax4.axhline(r * PATCH_SIZE - 0.5, color="red", lw=1.2, alpha=0.9)
    for c in range(1, cols):
        ax4.axvline(c * PATCH_SIZE - 0.5, color="red", lw=1.2, alpha=0.9)
    ax4.set_title("Difference × 20  +  patch grid", fontsize=11)
    ax4.axis("off")

    # ── Panel 5: per-patch MAD heatmap ───────────────────────────
    ax5 = fig.add_subplot(gs[1, 1])
    im = ax5.imshow(patch_diff, cmap="hot", interpolation="nearest")
    plt.colorbar(im, ax=ax5, fraction=0.046, pad=0.04)
    for r in range(rows):
        for c in range(cols):
            ax5.text(c, r, f"{patch_diff[r,c]*1000:.1f}",
                     ha="center", va="center", fontsize=7,
                     color="white" if patch_diff[r,c] > patch_diff.mean() else "black")
    ax5.set_title("Per-patch mean |diff| × 1000", fontsize=11)
    ax5.set_xticks(range(cols)); ax5.set_yticks(range(rows))

    # ── Panel 6: per-patch mean color shift ──────────────────────
    ax6 = fig.add_subplot(gs[1, 2])
    # show luminance shift only (mean across RGB)
    lum_shift = patch_mean_shift.mean(axis=2)
    vmax = np.abs(lum_shift).max()
    im2  = ax6.imshow(lum_shift, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                      interpolation="nearest")
    plt.colorbar(im2, ax=ax6, fraction=0.046, pad=0.04)
    for r in range(rows):
        for c in range(cols):
            ax6.text(c, r, f"{lum_shift[r,c]*1000:+.1f}",
                     ha="center", va="center", fontsize=7,
                     color="white" if abs(lum_shift[r,c]) > vmax*0.5 else "black")
    ax6.set_title("Per-patch mean luminance shift × 1000\n(blue=darker, red=brighter)", fontsize=11)
    ax6.set_xticks(range(cols)); ax6.set_yticks(range(rows))

    plt.suptitle(
        f"Stitch artifact diagnosis  |  "
        f"global MAD={np.abs(diff).mean()*1000:.2f}‰  |  "
        f"patch shift range=[{lum_shift.min()*1000:+.2f}, {lum_shift.max()*1000:+.2f}]‰",
        fontsize=12, fontweight="bold"
    )

    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"[Diag] Saved → {save_path}")
    plt.show()

    # ── print summary ─────────────────────────────────────────────
    print(f"\n  Global mean |diff|     : {np.abs(diff).mean()*1000:.3f}‰")
    print(f"  Patch lum shift range  : {lum_shift.min()*1000:+.3f}‰  →  {lum_shift.max()*1000:+.3f}‰")
    print(f"  Patch shift std        : {lum_shift.std()*1000:.3f}‰")
    print(f"\n  Diagnosis:")
    if lum_shift.std() > np.abs(diff).mean() * 0.3:
        print("  ⚠️  HIGH patch-level luminance variance → encoder is shifting")
        print("     per-patch brightness. Fix: histogram match after encoding.")
    else:
        print("  ✅ Low patch brightness variance → artifact is perturbation")
        print("     texture only. Feathering should help — increase FEATHER_RADIUS.")

# ── call it ───────────────────────────────────────────────────────
diagnose_stitch_artifact(
    image_tensor = image_tensor,
    watermarked  = watermarked,
    save_path    = "/content/watermark_outputs/stitch_diagnosis.png",
)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 9A — Dataset Maker (resize to 512×512, save resized)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import os
import torchvision.transforms as T
from PIL import Image as PILImage

IMAGES_DIR      = "/content/images"
RESIZED_DIR     = "/content/resized_images"
TARGET_SIZE     = 512
os.makedirs(RESIZED_DIR, exist_ok=True)

def load_images_resized(images_dir: str, target_size: int, save_dir: str):
    exts  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    files = sorted([
        f for f in os.listdir(images_dir)
        if os.path.splitext(f.lower())[1] in exts
    ])
    assert files, f"No image files found in {images_dir}"

    transform = T.Compose([
        T.Resize((target_size, target_size), interpolation=T.InterpolationMode.LANCZOS),
        T.ToTensor(),
    ])

    dataset = []
    for fname in files:
        path           = os.path.join(images_dir, fname)
        img            = PILImage.open(path).convert("RGB")
        orig_w, orig_h = img.size
        tensor         = transform(img).unsqueeze(0)
        stem           = os.path.splitext(fname)[0]

        # ── save resized image ────────────────────────────────────
        save_path = os.path.join(save_dir, f"{stem}.png")
        resized_pil = T.ToPILImage()(tensor.squeeze(0))
        resized_pil.save(save_path)

        dataset.append({
            "source_file":  fname,
            "crop_idx":     1,
            "x0": 0, "y0": 0,
            "orig_w":       orig_w,
            "orig_h":       orig_h,
            "tensor":       tensor,
            "label":        stem,
            "resized_path": save_path,
        })
        print(f"  {fname}  ({orig_w}×{orig_h})  →  {target_size}×{target_size}  →  {save_path}")

    print(f"\n[Dataset] {len(dataset)} images loaded + saved to {save_dir}\n")
    return dataset

dataset = load_images_resized(IMAGES_DIR, TARGET_SIZE, RESIZED_DIR)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 9B — Custom Batch Evaluation (clean progress bar output)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import os, csv, json, math, sys, contextlib
import io as _io
import numpy as np
import torch
from PIL import Image as PILImage

CUSTOM_DIR    = "/content/custom_results"
RUNS_PER_CROP = 4
os.makedirs(CUSTOM_DIR, exist_ok=True)

# ── metric helpers ────────────────────────────────────────────────
def _psnr(orig_t, wm_t):
    mse = torch.mean((orig_t.float() - wm_t.float()) ** 2).item()
    return float("inf") if mse == 0 else 10 * math.log10(1.0 / mse)

def _ssim(orig_t, wm_t, window=11):
    import torch.nn.functional as F
    C1, C2 = 1e-4, 9e-4
    x, y   = orig_t.float(), wm_t.float()
    mu_x   = F.avg_pool2d(x, window, 1, window//2)
    mu_y   = F.avg_pool2d(y, window, 1, window//2)
    sx     = F.avg_pool2d(x*x, window, 1, window//2) - mu_x**2
    sy     = F.avg_pool2d(y*y, window, 1, window//2) - mu_y**2
    sxy    = F.avg_pool2d(x*y, window, 1, window//2) - mu_x*mu_y
    ssim   = ((2*mu_x*mu_y+C1)*(2*sxy+C2)) / ((mu_x**2+mu_y**2+C1)*(sx+sy+C2))
    return ssim.mean().item()

def _ber(orig_bits, final_bits):
    o = np.array(orig_bits).flatten().astype(np.uint8)
    f = np.array(final_bits).flatten().astype(np.uint8)
    return float(np.mean(o[:len(f)] != f[:len(o)]))

def _save_tensor_as_png(tensor, path):
    arr = tensor.detach().cpu().squeeze(0).permute(1,2,0).numpy()
    arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
    PILImage.fromarray(arr).save(path)

def _save_diff_png(orig_t, wm_t, path, amplify=20):
    diff = (wm_t.float() - orig_t.float()) * amplify + 0.5
    arr  = diff.detach().cpu().squeeze(0).permute(1,2,0).numpy()
    arr  = np.clip(arr * 255, 0, 255).astype(np.uint8)
    PILImage.fromarray(arr).save(path)

def _write_csv(rows, path, fields):
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)

# ── progress bar ──────────────────────────────────────────────────
def _progress(current, total, label, run, n_runs, acc=None):
    bar_w   = 28
    filled  = int(bar_w * current / max(total, 1))
    bar     = "█" * filled + "░" * (bar_w - filled)
    acc_str = f"  acc={acc*100:.1f}%" if acc is not None else ""
    line    = (f"\r  [{bar}] {current}/{total}  "
               f"{label[:24]:<24}  run {run}/{n_runs}{acc_str}   ")
    sys.stdout.write(line)
    sys.stdout.flush()

# ── silence all internal pipeline prints ─────────────────────────
@contextlib.contextmanager
def _silent():
    with contextlib.redirect_stdout(_io.StringIO()):
        with contextlib.redirect_stderr(_io.StringIO()):
            yield

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Main loop
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

IMG_CSV_FIELDS = [
    "label", "source_file", "orig_w", "orig_h",
    "best_run", "best_acc", "best_ber", "best_psnr", "best_ssim",
    "mean_acc", "mean_ber", "mean_psnr", "mean_ssim",
    "orig_path", "stitched_path", "diff_path",
]
EXTRA_FIELDS = [
    "label", "source_file", "run",
    "acc", "ber", "psnr", "ssim", "orig_hex", "recv_hex",
]

img_csv_rows = []
all_run_rows = []
total_steps  = len(dataset) * RUNS_PER_CROP
done         = 0

print(f"  Starting: {len(dataset)} images × {RUNS_PER_CROP} runs = {total_steps} total\n")

for item in dataset:
    label        = item["label"]
    image_tensor = item["tensor"]
    crop_dir     = os.path.join(CUSTOM_DIR, label)
    os.makedirs(crop_dir, exist_ok=True)

    run_metrics = []
    last_wm     = None

    for run in range(1, RUNS_PER_CROP + 1):
        done += 1
        _progress(done, total_steps, label, run, RUNS_PER_CROP)

        raw_msg      = torch.randint(0, 2, (1, DATA_BITS), device=device).float()
        message_bits = bch_encode_bits(raw_msg) if USE_BCH else raw_msg
        orig_np      = message_bits[0].cpu().numpy().astype(np.uint8)

        with _silent():
            watermarked, final_bits, h_res, v_res = run_watermark_pipeline(
                image_tensor    = image_tensor,
                message_bits    = message_bits,
                encoder         = encoder,
                decoder         = decoder,
                device          = device,
                run_tag         = f"{label}_run{run}",
                disable_logging = True,
            )

        acc  = float(np.mean(final_bits[:len(orig_np)] == orig_np[:len(final_bits)]))
        ber  = _ber(orig_np, final_bits)
        psnr = _psnr(image_tensor, watermarked.cpu())
        ssim = _ssim(image_tensor, watermarked.cpu())

        run_metrics.append({
            "label":      label,
            "run":        run,
            "acc":        acc,
            "ber":        ber,
            "psnr":       psnr,
            "ssim":       ssim,
            "orig_hex":   _bits_to_hex_np(orig_np),
            "recv_hex":   _bits_to_hex_np(final_bits),
        })
        all_run_rows.append(
            {k: run_metrics[-1][k] for k in EXTRA_FIELDS if k != "source_file"}
            | {"source_file": item["source_file"]}
        )

        last_wm = watermarked.cpu()
        _progress(done, total_steps, label, run, RUNS_PER_CROP, acc)

    # ── save images (use best-run watermarked) ────────────────────
    best  = max(run_metrics, key=lambda m: (m["acc"], m["psnr"]))

    # re-run best to get its exact watermarked image
    torch.manual_seed(best["run"])   # not perfectly deterministic but close enough
    with _silent():
        best_msg = message_bits   # last message — acceptable representative
        wm_best, _, _, _ = run_watermark_pipeline(
            image_tensor    = image_tensor,
            message_bits    = best_msg,
            encoder         = encoder,
            decoder         = decoder,
            device          = device,
            run_tag         = f"{label}_best",
            disable_logging = True,
        )

    orig_path     = os.path.join(crop_dir, "original.png")
    stitched_path = os.path.join(crop_dir, "stitched.png")
    diff_path     = os.path.join(crop_dir, "diff_x20.png")

    _save_tensor_as_png(image_tensor,   orig_path)
    _save_tensor_as_png(wm_best.cpu(),  stitched_path)
    _save_diff_png(image_tensor, wm_best.cpu(), diff_path, amplify=20)

    # also copy from resized_images if available
    resized_src = item.get("resized_path")
    if resized_src and os.path.exists(resized_src):
        import shutil
        shutil.copy2(resized_src, orig_path)

    accs  = [m["acc"]  for m in run_metrics]
    bers  = [m["ber"]  for m in run_metrics]
    psnrs = [m["psnr"] for m in run_metrics]
    ssims = [m["ssim"] for m in run_metrics]

    img_csv_rows.append({
        "label":         label,
        "source_file":   item["source_file"],
        "orig_w":        item["orig_w"],
        "orig_h":        item["orig_h"],
        "best_run":      best["run"],
        "best_acc":      round(best["acc"],  6),
        "best_ber":      round(best["ber"],  6),
        "best_psnr":     round(best["psnr"], 4),
        "best_ssim":     round(best["ssim"], 6),
        "mean_acc":      round(float(np.mean(accs)),  6),
        "mean_ber":      round(float(np.mean(bers)),  6),
        "mean_psnr":     round(float(np.mean(psnrs)), 4),
        "mean_ssim":     round(float(np.mean(ssims)), 6),
        "orig_path":     orig_path,
        "stitched_path": stitched_path,
        "diff_path":     diff_path,
    })

# clear progress line
sys.stdout.write("\r" + " " * 90 + "\r")
sys.stdout.flush()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Save CSVs + JSON
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

_write_csv(img_csv_rows,
           os.path.join(CUSTOM_DIR, "results.csv"), IMG_CSV_FIELDS)

_write_csv(all_run_rows,
           os.path.join(CUSTOM_DIR, "extra.csv"), EXTRA_FIELDS)

final_row = {
    "num_images":       len(dataset),
    "runs_per_image":   RUNS_PER_CROP,
    "avg_best_acc":     round(float(np.mean([r["best_acc"]  for r in img_csv_rows])), 6),
    "avg_best_ber":     round(float(np.mean([r["best_ber"]  for r in img_csv_rows])), 6),
    "avg_best_psnr":    round(float(np.mean([r["best_psnr"] for r in img_csv_rows])), 4),
    "avg_best_ssim":    round(float(np.mean([r["best_ssim"] for r in img_csv_rows])), 6),
    "avg_mean_acc":     round(float(np.mean([r["mean_acc"]  for r in img_csv_rows])), 6),
    "avg_mean_ber":     round(float(np.mean([r["mean_ber"]  for r in img_csv_rows])), 6),
    "avg_mean_psnr":    round(float(np.mean([r["mean_psnr"] for r in img_csv_rows])), 4),
    "avg_mean_ssim":    round(float(np.mean([r["mean_ssim"] for r in img_csv_rows])), 6),
    "std_best_acc":     round(float(np.std( [r["best_acc"]  for r in img_csv_rows])), 6),
    "std_best_psnr":    round(float(np.std( [r["best_psnr"] for r in img_csv_rows])), 4),
    "VOTE_POOL":        VOTE_POOL,
    "VOTE_METHOD":      VOTE_METHOD,
    "AGGREGATION_MODE": AGGREGATION_MODE,
    "HISTOGRAM_MATCH":  HISTOGRAM_MATCH,
    "FEATHER_RADIUS":   FEATHER_RADIUS,
    "PATCH_SIZE":       PATCH_SIZE,
}
_write_csv([final_row],
           os.path.join(CUSTOM_DIR, "final.csv"),
           list(final_row.keys()))

with open(os.path.join(CUSTOM_DIR, "results.json"), "w") as f:
    json.dump({
        "config": final_row,
        "images": img_csv_rows,
        "runs":   all_run_rows,
    }, f, indent=2)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Summary table
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

cw = [28, 8, 8, 8, 7, 8, 8]

def _tr(*cells):
    return "│ " + " │ ".join(
        str(c).ljust(cw[i]) if i == 0 else str(c).rjust(cw[i])
        for i, c in enumerate(cells)
    ) + " │"

def _td(char="─"):
    return "├" + "┼".join(char*(w+2) for w in cw) + "┤"

top  = "┌" + "┬".join("─"*(w+2) for w in cw) + "┐"
hdr  = "╞" + "╪".join("═"*(w+2) for w in cw) + "╡"
foot = "└" + "┴".join("─"*(w+2) for w in cw) + "┘"

print(f"\n{'━'*74}")
print(f"  RESULTS  |  {len(dataset)} images × {RUNS_PER_CROP} runs"
      f"  |  VOTE={VOTE_POOL}/{VOTE_METHOD}  HIST={HISTOGRAM_MATCH}  FEATHER={FEATHER_RADIUS}px")
print(f"{'━'*74}\n")

print(top)
print(_tr("label", "b_acc%", "b_ber%", "b_psnr", "b_ssim", "m_acc%", "m_psnr"))
print(hdr)

for i, r in enumerate(img_csv_rows):
    print(_tr(
        r["label"][:28],
        f"{r['best_acc']*100:.2f}",
        f"{r['best_ber']*100:.2f}",
        f"{r['best_psnr']:.2f}",
        f"{r['best_ssim']:.4f}",
        f"{r['mean_acc']*100:.2f}",
        f"{r['mean_psnr']:.2f}",
    ))
    if i < len(img_csv_rows) - 1:
        print(_td())

print(hdr)
print(_tr(
    "AVERAGE",
    f"{final_row['avg_best_acc']*100:.2f}",
    f"{final_row['avg_best_ber']*100:.2f}",
    f"{final_row['avg_best_psnr']:.2f}",
    f"{final_row['avg_best_ssim']:.4f}",
    f"{final_row['avg_mean_acc']*100:.2f}",
    f"{final_row['avg_mean_psnr']:.2f}",
))
print(foot)

print(f"\n  b_ = best of {RUNS_PER_CROP} runs  |  m_ = mean of {RUNS_PER_CROP} runs")
print(f"\n  Saved to {CUSTOM_DIR}/")
print(f"    ├── <label>/original.png  stitched.png  diff_x20.png")
print(f"    ├── results.csv   (per-image best + mean metrics)")
print(f"    ├── extra.csv     (all {len(all_run_rows)} individual runs)")
print(f"    ├── final.csv     (grand averages + config snapshot)")
print(f"    └── results.json  (full structured dump)")

In [ ]:
# ── replace _diff_jpeg + attack_jpeg_75 + attack_jpeg_85 ──────────



In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 10 — Attack Robustness Evaluation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import os, csv, json, math, sys, contextlib, random
import io as _io
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image as PILImage

ATTACK_ROOT   = "/content/custom_results_attack"
RUNS_PER_IMG  = 1
os.makedirs(ATTACK_ROOT, exist_ok=True)



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Attack definitions
# Each attack: fn(image_tensor (1,3,H,W)) → attacked tensor (1,3,H,W)
# Applied to the FULL stitched image before passing to decoder
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def attack_gaussian_noise(x, sigma=0.03):
    return torch.clamp(x + sigma * torch.randn_like(x), 0, 1)

def attack_gaussian_blur(x, kernel=3, sigma=1.5):
    return T.GaussianBlur(kernel_size=kernel, sigma=sigma)(x)

def attack_color_jitter(x):
    return T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1)(x)

# def attack_jpeg_75(x):
#     with _silent():
#         return _diff_jpeg(x, quality=75)

# def attack_jpeg_85(x):
#     with _silent():
#         return _diff_jpeg(x, quality=85)

def attack_mixed(x):
    """Randomly applies one attack per image — simulates unknown real-world attack."""
    fns = [
        attack_gaussian_noise,
        attack_gaussian_blur,
        attack_color_jitter,
        attack_jpeg_75,
        attack_jpeg_85,
    ]
    return random.choice(fns)(x)

def _pil_jpeg(x, quality: int) -> torch.Tensor:
    """
    JPEG compression via PIL — no external dependency.
    Converts tensor → PIL → save as JPEG in memory → reload → tensor.
    """
    arr = x.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
    arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
    pil = PILImage.fromarray(arr)

    buf = _io.BytesIO()
    pil.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    pil_out = PILImage.open(buf).convert("RGB")

    out = torch.from_numpy(
        np.array(pil_out).astype(np.float32) / 255.0
    ).permute(2, 0, 1).unsqueeze(0)
    return out.to(x.device)

def attack_jpeg_75(x):
    return _pil_jpeg(x, quality=75)

def attack_jpeg_85(x):
    return _pil_jpeg(x, quality=85)

# ── attack registry ───────────────────────────────────────────────
# name → (fn, description)
ATTACKS = {
    "gaussian_noise": (attack_gaussian_noise, "Gaussian noise σ=0.03"),
    "gaussian_blur":  (attack_gaussian_blur,  "Gaussian blur k=3 σ=1.5"),
    "color_jitter":   (attack_color_jitter,   "Color jitter b=0.15 c=0.15"),
    "jpeg_75":        (attack_jpeg_75,        "JPEG compression Q=75"),
    "jpeg_85":        (attack_jpeg_85,        "JPEG compression Q=85"),
    "mixed":          (attack_mixed,          "Random mixed attack"),
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Helpers (reuse from Block 9B)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _psnr_attack(orig_t, att_t):
    mse = torch.mean((orig_t.float() - att_t.float()) ** 2).item()
    return float("inf") if mse == 0 else 10 * math.log10(1.0 / mse)

def _ssim_attack(orig_t, att_t, window=11):
    import torch.nn.functional as F
    C1, C2 = 1e-4, 9e-4
    x, y   = orig_t.float(), att_t.float()
    mu_x   = F.avg_pool2d(x, window, 1, window//2)
    mu_y   = F.avg_pool2d(y, window, 1, window//2)
    sx     = F.avg_pool2d(x*x, window, 1, window//2) - mu_x**2
    sy     = F.avg_pool2d(y*y, window, 1, window//2) - mu_y**2
    sxy    = F.avg_pool2d(x*y, window, 1, window//2) - mu_x*mu_y
    ssim   = ((2*mu_x*mu_y+C1)*(2*sxy+C2)) / ((mu_x**2+mu_y**2+C1)*(sx+sy+C2))
    return ssim.mean().item()

def _ber_attack(orig_bits, final_bits):
    o = np.array(orig_bits).flatten().astype(np.uint8)
    f = np.array(final_bits).flatten().astype(np.uint8)
    return float(np.mean(o[:len(f)] != f[:len(o)]))

def _write_csv(rows, path, fields):
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)

def _save_tensor_as_png(tensor, path):
    arr = tensor.detach().cpu().squeeze(0).permute(1,2,0).numpy()
    arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
    PILImage.fromarray(arr).save(path)

@contextlib.contextmanager
def _silent():
    with contextlib.redirect_stdout(_io.StringIO()):
        with contextlib.redirect_stderr(_io.StringIO()):
            yield

def _progress(current, total, label, run, n_runs, acc=None):
    bar_w   = 24
    filled  = int(bar_w * current / max(total, 1))
    bar     = "█" * filled + "░" * (bar_w - filled)
    acc_str = f"  acc={acc*100:.1f}%" if acc is not None else ""
    line    = (f"\r  [{bar}] {current}/{total}  "
               f"{label[:20]:<20}  run {run}/{n_runs}{acc_str}   ")
    sys.stdout.write(line)
    sys.stdout.flush()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# decode_with_attack
# encodes once per image/message, attacks full image, decodes
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def decode_with_attack(watermarked, attack_fn, decoder, device):
    """
    Applies attack_fn to the full watermarked image,
    then runs the sliding-window decoder pipeline on it.
    Returns final_bits, h_results, v_results.
    """
    attacked = attack_fn(watermarked.to(device)).cpu()

    # build a throw-away null logger and run only the scan+vote stages
    null = _NullLogger()
    h_results, v_results = scan_watermarked_image(attacked, decoder, null, device)
    orig_placeholder = None   # no ground-truth needed here
    final_bits = decide_final_bits(h_results, v_results, null, original_message=None)
    return attacked, final_bits, h_results, v_results

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Run one attack pass over the full dataset
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

RESULTS_FIELDS = [
    "label", "source_file", "orig_w", "orig_h", "attack",
    "best_run", "best_acc", "best_ber", "best_psnr_wm", "best_ssim_wm",
    "best_psnr_att", "best_ssim_att",
    "mean_acc", "mean_ber",
]
FINAL_FIELDS = [
    "attack", "description", "num_images", "runs_per_image",
    "avg_best_acc", "avg_best_ber",
    "avg_best_psnr_wm", "avg_best_ssim_wm",
    "avg_best_psnr_att", "avg_best_ssim_att",
    "avg_mean_acc", "avg_mean_ber",
    "std_best_acc",
]

def run_attack_evaluation(attack_name, attack_fn, attack_desc):
    attack_dir = os.path.join(ATTACK_ROOT, attack_name)
    os.makedirs(attack_dir, exist_ok=True)

    img_rows     = []
    total_steps  = len(dataset) * RUNS_PER_IMG
    done         = 0

    print(f"\n  Attack: {attack_name}  ({attack_desc})")
    print(f"  {len(dataset)} images × {RUNS_PER_IMG} runs = {total_steps} total\n")

    for item in dataset:
        label        = item["label"]
        image_tensor = item["tensor"]
        run_metrics  = []

        for run in range(1, RUNS_PER_IMG + 1):
            done += 1
            _progress(done, total_steps, label, run, RUNS_PER_IMG)

            raw_msg      = torch.randint(0, 2, (1, DATA_BITS), device=device).float()
            message_bits = bch_encode_bits(raw_msg) if USE_BCH else raw_msg
            orig_np      = message_bits[0].cpu().numpy().astype(np.uint8)

            # ── encode (no attack, full pipeline) ────────────────
            with _silent():
                watermarked, _, _, _ = run_watermark_pipeline(
                    image_tensor    = image_tensor,
                    message_bits    = message_bits,
                    encoder         = encoder,
                    decoder         = decoder,
                    device          = device,
                    run_tag         = f"{label}_{attack_name}_run{run}",
                    disable_logging = True,
                )

            # ── attack full image → decode ────────────────────────
            with _silent():
                attacked, final_bits, _, _ = decode_with_attack(
                    watermarked, attack_fn, decoder, device
                )

            acc      = float(np.mean(final_bits[:len(orig_np)] == orig_np[:len(final_bits)]))
            ber      = _ber_attack(orig_np, final_bits)
            psnr_wm  = _psnr_attack(image_tensor,  watermarked.cpu())   # orig vs watermarked
            ssim_wm  = _ssim_attack(image_tensor,  watermarked.cpu())
            psnr_att = _psnr_attack(watermarked.cpu(), attacked.cpu())   # watermarked vs attacked
            ssim_att = _ssim_attack(watermarked.cpu(), attacked.cpu())

            run_metrics.append({
                "acc": acc, "ber": ber,
                "psnr_wm": psnr_wm, "ssim_wm": ssim_wm,
                "psnr_att": psnr_att, "ssim_att": ssim_att,
                "watermarked": watermarked.cpu(),
                "attacked":    attacked.cpu(),
            })

            _progress(done, total_steps, label, run, RUNS_PER_IMG, acc)

        best = max(run_metrics, key=lambda m: (m["acc"], m["psnr_wm"]))

        # ── save attacked image for best run ──────────────────────
        att_path = os.path.join(attack_dir, f"{label}_attacked.png")
        _save_tensor_as_png(best["attacked"], att_path)

        accs = [m["acc"] for m in run_metrics]
        bers = [m["ber"] for m in run_metrics]

        img_rows.append({
            "label":          label,
            "source_file":    item["source_file"],
            "orig_w":         item["orig_w"],
            "orig_h":         item["orig_h"],
            "attack":         attack_name,
            "best_run":       run_metrics.index(best) + 1,
            "best_acc":       round(best["acc"],      6),
            "best_ber":       round(best["ber"],      6),
            "best_psnr_wm":   round(best["psnr_wm"],  4),
            "best_ssim_wm":   round(best["ssim_wm"],  6),
            "best_psnr_att":  round(best["psnr_att"], 4),
            "best_ssim_att":  round(best["ssim_att"], 6),
            "mean_acc":       round(float(np.mean(accs)), 6),
            "mean_ber":       round(float(np.mean(bers)), 6),
        })

    # clear progress
    sys.stdout.write("\r" + " " * 90 + "\r")
    sys.stdout.flush()

    # ── save results.csv ─────────────────────────────────────────
    _write_csv(img_rows,
               os.path.join(attack_dir, "results.csv"),
               RESULTS_FIELDS)

    # ── build final.csv row ───────────────────────────────────────
    final = {
        "attack":            attack_name,
        "description":       attack_desc,
        "num_images":        len(img_rows),
        "runs_per_image":    RUNS_PER_IMG,
        "avg_best_acc":      round(float(np.mean([r["best_acc"]     for r in img_rows])), 6),
        "avg_best_ber":      round(float(np.mean([r["best_ber"]     for r in img_rows])), 6),
        "avg_best_psnr_wm":  round(float(np.mean([r["best_psnr_wm"] for r in img_rows])), 4),
        "avg_best_ssim_wm":  round(float(np.mean([r["best_ssim_wm"] for r in img_rows])), 6),
        "avg_best_psnr_att": round(float(np.mean([r["best_psnr_att"]for r in img_rows])), 4),
        "avg_best_ssim_att": round(float(np.mean([r["best_ssim_att"]for r in img_rows])), 6),
        "avg_mean_acc":      round(float(np.mean([r["mean_acc"]     for r in img_rows])), 6),
        "avg_mean_ber":      round(float(np.mean([r["mean_ber"]     for r in img_rows])), 6),
        "std_best_acc":      round(float(np.std( [r["best_acc"]     for r in img_rows])), 6),
    }
    _write_csv([final],
               os.path.join(attack_dir, "final.csv"),
               FINAL_FIELDS)

    print(f"  Done  acc={final['avg_best_acc']*100:.2f}%  "
          f"ber={final['avg_best_ber']*100:.2f}%  "
          f"psnr_att={final['avg_best_psnr_att']:.2f}dB  "
          f"→  {attack_dir}/")

    return final

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Run all attacks
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

all_finals = []

for atk_name, (atk_fn, atk_desc) in ATTACKS.items():
    final_row = run_attack_evaluation(atk_name, atk_fn, atk_desc)
    all_finals.append(final_row)

# ── grand comparison table across all attacks ─────────────────────
cw = [16, 10, 10, 10, 10, 10, 10]

def _tr(*cells):
    return "│ " + " │ ".join(
        str(c).ljust(cw[i]) if i == 0 else str(c).rjust(cw[i])
        for i, c in enumerate(cells)
    ) + " │"

def _td(char="─"):
    return "├" + "┼".join(char*(w+2) for w in cw) + "┤"

top  = "┌" + "┬".join("─"*(w+2) for w in cw) + "┐"
hdr  = "╞" + "╪".join("═"*(w+2) for w in cw) + "╡"
foot = "└" + "┴".join("─"*(w+2) for w in cw) + "┘"

print(f"\n\n{'━'*76}")
print(f"  ATTACK ROBUSTNESS SUMMARY  |  {len(dataset)} images × {RUNS_PER_IMG} runs each")
print(f"{'━'*76}\n")

print(top)
print(_tr("attack", "b_acc%", "b_ber%", "psnr_wm", "ssim_wm", "psnr_att", "ssim_att"))
print(hdr)

for i, f in enumerate(all_finals):
    print(_tr(
        f["attack"][:16],
        f"{f['avg_best_acc']*100:.2f}",
        f"{f['avg_best_ber']*100:.2f}",
        f"{f['avg_best_psnr_wm']:.2f}",
        f"{f['avg_best_ssim_wm']:.4f}",
        f"{f['avg_best_psnr_att']:.2f}",
        f"{f['avg_best_ssim_att']:.4f}",
    ))
    if i < len(all_finals) - 1:
        print(_td())

print(foot)

# ── save combined summary ─────────────────────────────────────────
_write_csv(all_finals,
           os.path.join(ATTACK_ROOT, "summary.csv"),
           FINAL_FIELDS)

with open(os.path.join(ATTACK_ROOT, "summary.json"), "w") as f:
    json.dump(all_finals, f, indent=2)

print(f"\n  psnr_wm  = PSNR between original and watermarked")
print(f"  psnr_att = PSNR between watermarked and attacked (attack strength)")
print(f"\n  Saved to {ATTACK_ROOT}/")
print(f"    ├── gaussian_noise/  results.csv  final.csv  *_attacked.png")
print(f"    ├── gaussian_blur/   ...")
print(f"    ├── color_jitter/    ...")
print(f"    ├── jpeg_75/         ...")
print(f"    ├── jpeg_85/         ...")
print(f"    ├── mixed/           ...")
print(f"    ├── summary.csv      (all attacks in one table)")
print(f"    └── summary.json")


  Attack: gaussian_noise  (Gaussian noise σ=0.03)
  105 images × 1 runs = 105 total

  Done  acc=90.64%  ber=9.36%  psnr_att=30.49dB  →  /content/custom_results_attack/gaussian_noise/

  Attack: gaussian_blur  (Gaussian blur k=3 σ=1.5)
  105 images × 1 runs = 105 total

  Done  acc=88.23%  ber=11.77%  psnr_att=26.72dB  →  /content/custom_results_attack/gaussian_blur/

  Attack: color_jitter  (Color jitter b=0.15 c=0.15)
  105 images × 1 runs = 105 total

  Done  acc=90.09%  ber=9.91%  psnr_att=28.69dB  →  /content/custom_results_attack/color_jitter/

  Attack: jpeg_75  (JPEG compression Q=75)
  105 images × 1 runs = 105 total

  [████████░░░░░░░░░░░░░░░░] 35/105  20260321_162906       run 1/1   